In [2]:
# NPGA (ES gradients) with phi sampled from Drive CSV
# Two payment scenarios:
#   (A) Uniform market-clearing price (all winners paid p* = marginal accepted bid)
#   (B) Pay-as-bid (each winner paid own bid)
# Allocation: accept lowest bids until demand is met; capacity per supplier; bids above price cap are rejected.
# Utility: u_i = q_i_alloc * (payment_i - v_i) if accepted else 0.

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import ast
import torch.nn.functional as F
from typing import Dict, Any
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
#from google.colab import drive
import ast
import torch.nn.functional as F
from typing import Dict, Any
from typing import Dict, Any, List, Optional, Tuple
import numpy as np
import pandas as pd
import ast

In [3]:
import sys, torch
print(sys.version)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
2.5.1+cu121
12.1
True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


## PARTE 0: CARGAR DATOS

In [4]:

# =========================
# 0) Load CSV from Drive
# =========================
#drive.mount("/content/drive")
CSV_PATH = "../../results/fechas_117/TERMICA/inputRed.csv"   # <-- adjust
df = pd.read_csv(
    CSV_PATH,
    sep=";",
    decimal=",",      # 817128,3029 -> 817128.3029
    thousands="."     # si tienes 1.234.567,89
)
required = ['id_planta', "available_q", "phi", "precio_d", "daily_reman_demand"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

# Optional: if present, enables per-player phi sampling
AGENT_COL = "id_planta" if "id_planta" in df.columns else None


# =========================
# 1) Device / seeds
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(">>> Running on device:", device)

torch.manual_seed(0)
np.random.seed(0)

TensorDict = Dict[str, torch.Tensor]

>>> Running on device: cuda


In [5]:
df

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster,id_planta
0,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",830.944358,889.28,"[238087.97000000067, 1032624.4100000001, 83845...",2025-04-01,10.0,1
1,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",826.880073,889.28,"[233350.0, 205300.0, 951070.0, 872530.0, 10147...",2025-04-02,13.0,1
2,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",828.001249,889.28,"[895720.0, 609450.0, 422830.0, 348240.0, 55249...",2025-04-03,7.0,1
3,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",695.986348,889.28,"[270253.61000000034, 466977.03000000026, 23921...",2025-04-04,2.0,1
4,3ENA,"[200000.0, 200000.0, 200000.0, 200000.0, 20000...",767.694311,889.28,"[530630.0, 268940.0, 104170.0, 406420.0, 39726...",2025-04-07,5.0,1
...,...,...,...,...,...,...,...,...
4207,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",247.517152,338.94,"[907640.0, 634130.0, 569640.0, 339450.0, 34217...",2025-09-03,15.0,36
4208,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",193.320580,338.94,"[487560.0, 261410.0, 324070.0, 304360.0, 36979...",2025-09-04,9.0,36
4209,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",263.392797,338.94,"[957600.0, 748720.0, 575140.0, 465820.0, 67637...",2025-09-05,2.0,36
4210,ZPA5,"[63000.0, 63000.0, 63000.0, 63000.0, 63000.0, ...",139.564880,338.94,"[96490.0, 92810.0, 456530.0, 289440.0, 90450.0...",2025-09-08,5.0,36


In [6]:
def get_torch_device(device: str = "auto") -> torch.device:
    """
    device:
      - "auto": cuda si disponible, si no cpu
      - "cuda": fuerza cuda (error si no hay)
      - "cpu": fuerza cpu
      - "cuda:0", "cuda:1", etc.
    """
    device = (device or "auto").lower().strip()
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device)

In [7]:
# cantidad estan en kwh, lo paso a MWh

In [8]:
import numpy as np
import pandas as pd
import ast

SCALE = 1000.0

def div1000_list_string(x, scale=SCALE):
    # deja la columna como STRING tipo "[..., ...]" (igual estilo CSV)
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return np.nan

    # si viene como string "[...]" -> parse
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return None
        # si aparecen 'nan' en el string, literal_eval falla
        s = s.replace("nan", "None").replace("NaN", "None").replace("NAN", "None")
        L = ast.literal_eval(s)
    else:
        L = x  # ya es lista/array

    scaled = []
    for v in L:
        if v is None:
            scaled.append(None)
        else:
            fv = float(v)
            if np.isnan(fv):
                scaled.append(None)
            else:
                scaled.append(fv / scale)

    return str(scaled)  # vuelve a STRING con corchetes y comas

df = df.copy()

# listas como STRING (misma forma que originalmente)
df["available_q"] = df["available_q"].apply(div1000_list_string)
df["daily_reman_demand"] = df["daily_reman_demand"].apply(div1000_list_string)

# escalares (numéricos)
df["phi"] = pd.to_numeric(df["phi"], errors="coerce") / SCALE
df["precio_d"] = pd.to_numeric(df["precio_d"], errors="coerce") / SCALE


In [9]:
df

,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster,id_planta
0,3ENA,"[200.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200...",0.830944,0.88928,"[238.08797000000067, 1032.6244100000001, 838.4...",2025-04-01,10.0,1
1,3ENA,"[200.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200...",0.826880,0.88928,"[233.35, 205.3, 951.07, 872.53, 1014.76, 207.0...",2025-04-02,13.0,1
2,3ENA,"[200.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200...",0.828001,0.88928,"[895.72, 609.45, 422.83, 348.24, 552.49, 986.5...",2025-04-03,7.0,1
3,3ENA,"[200.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200...",0.695986,0.88928,"[270.2536100000003, 466.97703000000024, 239.21...",2025-04-04,2.0,1
4,3ENA,"[200.0, 200.0, 200.0, 200.0, 200.0, 200.0, 200...",0.767694,0.88928,"[530.63, 268.94, 104.17, 406.42, 397.26, 105.4...",2025-04-07,5.0,1
...,...,...,...,...,...,...,...,...
4207,ZPA5,"[63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63....",0.247517,0.33894,"[907.64, 634.13, 569.64, 339.45, 342.17, 740.6...",2025-09-03,15.0,36
4208,ZPA5,"[63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 5.25, 4.2...",0.193321,0.33894,"[487.56, 261.41, 324.07, 304.36, 369.79, 665.1...",2025-09-04,9.0,36
4209,ZPA5,"[63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63....",0.263393,0.33894,"[957.6, 748.72, 575.14, 465.82, 676.37, 903.96...",2025-09-05,2.0,36
4210,ZPA5,"[63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63.0, 63....",0.139565,0.33894,"[96.49, 92.81, 456.53, 289.44, 90.45, 838.61, ...",2025-09-08,5.0,36


In [10]:
df_nan = df[df['cluster'].isna()]
df_nan



,CodigoPlanta,available_q,phi,precio_d,daily_reman_demand,Fecha,cluster,id_planta
132,CTG1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,0.40737,"[934.56, 680.36, 498.99, 583.7, 729.55, 1168.1...",2025-04-23,NaN,2
192,CTG1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707000,0.71452,"[452.23, 184.66, 498.9, 383.91, 557.98, 441.6,...",2025-07-15,NaN,2
193,CTG1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.346328,0.34639,"[85.87, 462.86, 222.53, 166.18, 333.12, 72.86,...",2025-07-16,NaN,2
194,CTG1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.924574,0.95504,"[85.14, 265.74, 391.92, 291.44, 416.48, 359.22...",2025-07-17,NaN,2
195,CTG1,"[92.0, 92.0, 92.0, 92.0, 92.0, 92.0, 92.0, 92....",1.960851,1.96601,"[169.72, 228.91, 483.12, 367.63, 86.8880099999...",2025-07-18,NaN,2
...,...,...,...,...,...,...,...,...
3894,ZPA3,"[160.0, 160.0, 160.0, 160.0, 160.0, 160.0, 160...",0.946633,1.07243,"[144.69, 230.69, 54.85, 167.49, 239.03, 268.03...",2025-05-19,NaN,34
3898,ZPA3,"[46.0, 46.0, 46.0, 46.0, 46.0, 46.0, 46.0, 46....",0.867795,0.89962,"[126.32, 205.62, 1233.4, 1132.56, 1298.66, 114...",2025-05-23,NaN,34
4077,ZPA4,"[92.0, 92.0, 92.0, 92.0, 92.0, 92.0, 92.0, 92....",1.911497,1.96335,"[421.77758000000006, 506.8954299999997, 317.93...",2025-08-18,NaN,35
4078,ZPA4,"[47.0, 47.0, 47.0, 47.0, 47.0, 47.0, 47.0, 0.0...",2.203589,2.21375,"[1267.81, 927.43, 762.03, 720.18, 904.04, 1254...",2025-08-19,NaN,35


## PARTE 1: SAMPLE BATCHES OF VALUATIONS

In [11]:
from typing import Dict, Any, List, Optional, Tuple
import numpy as np
import pandas as pd
import ast

def _to_24(x) -> np.ndarray:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.full(24, np.nan, dtype=float)

    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            x = ast.literal_eval(s)
        else:
            return np.full(24, float(s), dtype=float)

    if isinstance(x, (list, tuple, np.ndarray)):
        a = np.asarray(x, dtype=float).reshape(-1)
        if a.size == 24:
            return a
        if a.size == 1:
            return np.full(24, float(a[0]), dtype=float)
        raise ValueError(f"Se esperaba len 24 o escalar; got len={a.size}")

    return np.full(24, float(x), dtype=float)


def sample_day_batches_phi(
    df: pd.DataFrame,
    *,
    H: int,
    seed: int = 123,
    day_col: str = "dia",            # fecha/índice de día
    id_col: str = "id_planta",
    phi_col: str = "phi",
    price_col: Optional[str] = "precio_d",   # usado solo para detectar inactivos (phi=0 & precio=0)
    plant_ids: Optional[List[Any]] = None,
    replace: bool = False,
    min_active_plants: int = 1,      # filtra días con al menos este número de plantas activas
    pick: str = "random",            # si hay múltiples filas por (día,planta): "random" o "first"
) -> Tuple[np.ndarray, Dict[Any, np.ndarray], np.ndarray]:
    """
    Cada h es un día. Retorna:
      - days_sampled: (H,)
      - phi_batch_by_plant[pid]: (H,24)  (NaN si planta inactiva ese día)
      - active_mask: (H,I) bool
      -aunque verifico dias activos e inactivos, al final supuse que todas las plantas estaban activas en todos los dias
    """
    for c in (day_col, id_col, phi_col):
        if c not in df.columns:
            raise ValueError(f"df debe tener columna '{c}'")

    if plant_ids is None:
        plant_ids = list(pd.unique(df[id_col].values))

    d = df.loc[df[id_col].isin(plant_ids), [day_col, id_col, phi_col] + ([price_col] if (price_col and price_col in df.columns) else [])].copy()

    # activo: por defecto, phi no NaN
    active = np.isfinite(pd.to_numeric(d[phi_col], errors="coerce"))

    # si tienes regla de ceros-sentinela: (phi==0 & precio==0) => inactivo
    if price_col and price_col in d.columns:
        phi_num = pd.to_numeric(d[phi_col], errors="coerce")
        price_num = pd.to_numeric(d[price_col], errors="coerce")
        inactive_zero = (phi_num == 0) & (price_num == 0)
        active = active & (~inactive_zero)

    d["active"] = active

    # días elegibles: al menos min_active_plants activas
    active_counts = (
        d.loc[d["active"], [day_col, id_col]]
         .drop_duplicates()
         .groupby(day_col)[id_col]
         .nunique()
    )
    eligible_days = active_counts.index[active_counts.values >= int(min_active_plants)].to_numpy()
    if eligible_days.size == 0:
        raise ValueError("No hay días elegibles con el mínimo de plantas activas.")

    rng = np.random.default_rng(int(seed))
    days_sampled = rng.choice(eligible_days, size=int(H), replace=replace)

    I = len(plant_ids)
    phi_batch_by_plant: Dict[Any, np.ndarray] = {pid: np.full((H, 24), np.nan, dtype=float) for pid in plant_ids}
    active_mask = np.zeros((H, I), dtype=bool)

    # para lookup rápido
    d2 = d.drop_duplicates(subset=[day_col, id_col], keep="first") if pick == "first" else d

    for h_idx, day in enumerate(days_sampled):
        d_day = d2.loc[d2[day_col] == day]

        for j, pid in enumerate(plant_ids):
            rows = d_day.loc[d_day[id_col] == pid]
            if len(rows) == 0:
                continue

            if pick == "random" and len(rows) > 1:
                row = rows.iloc[int(rng.integers(0, len(rows)))]
            else:
                row = rows.iloc[0]

            if bool(row["active"]):
                active_mask[h_idx, j] = True
                phi_batch_by_plant[pid][h_idx, :] = _to_24(row[phi_col])

    return  phi_batch_by_plant


## PARTE 2: CALCULAR EQUILIBRIOS

### PARTE 2.1 OBTENER BIDS

In [12]:
import numpy as np
import torch


class PolicyNet(nn.Module):
    def __init__(self, input_dim: int, hidden=(32, 64, 128, 128, 64, 32), output_dim: int = 1):
        super().__init__()
        layers = []
        d = int(input_dim)
        for h in hidden:
            layers += [nn.Linear(d, int(h)), nn.ReLU()]
            d = int(h)
        layers += [nn.Linear(d, int(output_dim))]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

import numpy as np
import pandas as pd
import torch

def init_theta0_by_pid_from_df(
    df,
    *,
    id_col="id_planta",
    phi_col="phi",   # <-- usa phi para b0
    price_col="precio_d",
    input_dim=1,
    hidden=(32,64,128,128,64,32),
    device = device,
    b0_mode="phi_median",          # "phi_median" o "phi_mean"
    money_scale=1.0,               # 1000.0 si necesitas forzar escala aquí
):
    if id_col not in df.columns:
        raise KeyError(f"df no tiene {id_col}")
    if phi_col not in df.columns:
        raise KeyError(f"df no tiene {phi_col}")

    d = df[[id_col, phi_col]].dropna().copy()
    d[phi_col] = pd.to_numeric(d[phi_col], errors="coerce")
    d = d.dropna(subset=[phi_col])

    # fuerza escala monetaria consistente si aplica
    d[phi_col] = d[phi_col] / float(money_scale)

    if b0_mode == "phi_median":
        b0_by_pid = d.groupby(id_col)[phi_col].median().to_dict()
    elif b0_mode == "phi_mean":
        b0_by_pid = d.groupby(id_col)[phi_col].mean().to_dict()
    else:
        raise ValueError("b0_mode debe ser 'phi_median' o 'phi_mean'")

    theta_by_pid = {}
    L = len(hidden)
    last_bias_key = f"net.{2*L}.bias"

    for pid, b0 in b0_by_pid.items():
        model = PolicyNet(input_dim=input_dim, hidden=hidden, output_dim=1).to(device)
        sd = model.state_dict()

        # Opción estable: todo en 0 y bias final = b0 (red constante al inicio)
        for k, v in sd.items():
            sd[k] = torch.zeros_like(v, device=device)

        sd[last_bias_key] = torch.tensor([float(b0)], dtype=sd[last_bias_key].dtype, device=device)

        theta_by_pid[int(pid) if isinstance(pid, (np.integer,)) else pid] = sd

    return theta_by_pid



In [13]:
import numpy as np
import torch
import torch.nn.functional as F
from typing import Dict, Any, Tuple

# ============================================================
# Helper: piso (phi) por escenario h, a partir de obs_batch
# - si obs es (H,)         -> floor = obs
# - si obs es (H,d)        -> floor = max/mean por fila
# ============================================================
def _phi_floor_from_obs(obs: np.ndarray, floor_mode: str = "max") -> np.ndarray:
    obs = np.asarray(obs, dtype=np.float32)
    if obs.ndim == 1:
        return obs.astype(float)  # (H,)
    if obs.ndim == 2:
        if floor_mode == "max":
            return obs.max(axis=1).astype(float)   # (H,)
        if floor_mode == "mean":
            return obs.mean(axis=1).astype(float)  # (H,)
        raise ValueError("floor_mode must be 'max' or 'mean'")
    raise ValueError(f"obs shape no soportado: {obs.shape}")


# ---------------------------------------------------------
# 3) Build nets per player + load θ, and compute β(o^h)
#    CON RESTRICCIÓN: beta(h) > phi_floor(h) y beta(h) > 0
# ---------------------------------------------------------
import numpy as np
import torch
import torch.nn.functional as F


def beta_matrix_from_obs_batch(
    nets_by_pid,
    obs_batch_by_pid,
    *,
    device="auto",
    enforce_beta_ge_phi=True,
    min_margin=1e-6,
    floor_mode="max",
    price_cap=None,
    return_numpy: bool = True,   # True -> np.ndarray, False -> torch.Tensor
):
    dev = _resolve_device(device)

    pids = list(nets_by_pid.keys())
    H = int(np.asarray(obs_batch_by_pid[pids[0]]).shape[0])
    I = len(pids)

    # Mantén Beta en GPU/CPU (torch) durante el loop
    Beta_t = torch.empty((H, I), dtype=torch.float32, device=dev)

    for j, pid in enumerate(pids):
        net = nets_by_pid[pid].to(dev)
        net.eval()

        obs_np = np.asarray(obs_batch_by_pid[pid], dtype=np.float32)

        if obs_np.ndim == 1:
            X = torch.as_tensor(obs_np.reshape(-1, 1), dtype=torch.float32, device=dev)
        elif obs_np.ndim == 2:
            X = torch.as_tensor(obs_np, dtype=torch.float32, device=dev)
        else:
            raise ValueError(f"obs_batch_by_pid[{pid}] shape no soportado: {obs_np.shape}")

        with torch.no_grad():
            raw = net(X).reshape(-1)  # bid en nivel
            b = raw

            if enforce_beta_ge_phi:
                phi_floor = _phi_floor_from_obs(obs_np, floor_mode=floor_mode)  # numpy (H,)
                phi_floor_t = torch.as_tensor(phi_floor, dtype=torch.float32, device=dev)
                b = torch.maximum(b, phi_floor_t + float(min_margin))

            if price_cap is not None:
                b = torch.minimum(b, torch.tensor(float(price_cap), dtype=torch.float32, device=dev))

            Beta_t[:, j] = b

    if return_numpy:
        # Convertir UNA sola vez al final
        return Beta_t.detach().cpu().numpy().astype(float), pids
    else:
        return Beta_t, pids


from typing import Dict, Any

def build_nets_by_pid(
    theta_by_pid: Dict[Any, Dict[str, torch.Tensor]],
    *,
    input_dim: int,
    hidden=(32, 64, 128, 128, 64, 32),
    device="auto",
):
    """
    Input:
      theta_by_pid: dict[pid] -> state_dict compatible con PolicyNet
    Output:
      nets_by_pid: dict[pid] -> PolicyNet en device, eval mode
    """
    nets_by_pid = {}
    for pid, sd in theta_by_pid.items():
        net = PolicyNet(input_dim=input_dim, hidden=hidden, output_dim=1).to(device)
        # por si el sd viene en cpu / otro device
        sd2 = {k: v.to(device) for k, v in sd.items()}
        net.load_state_dict(sd2, strict=True)
        net.eval()
        nets_by_pid[pid] = net
    return nets_by_pid


# -------------------------------------------------------------------
# 4) Entry point: given t, return θ, nets, and β_{t-1}(o^h)
#    CON RESTRICCIÓN beta > phi
# -------------------------------------------------------------------
def policies_and_beta_profile(
    *,
    t: int,
    df: pd.DataFrame,
    obs_batch_by_pid: Dict[Any, np.ndarray],           # o_i^h (en tu caso o=phi)
    theta_prev_by_pid: Dict[Any, TensorDict] = None,
    id_col: str = "id_planta",
    price_col: str = "precio_d",
    input_dim: int = 1,
    hidden=(32, 64, 128, 128, 64, 32),
    device: str = "auto",
    enforce_beta_gt_phi: bool = True,                  # <-- NUEVO
    min_margin: float = 1e-6,                          # <-- NUEVO
    floor_mode: str = "max",                           # <-- NUEVO
) -> dict:
    """
    Devuelve:
      {
        "theta_by_pid": θ usados,
        "nets_by_pid":  redes por jugador con esos θ,
        "Beta":         (H,I) con β(o^h) (cumple β>φ si enforce_beta_gt_phi=True),
        "pids":         orden columnas,
      }
    """
    if int(t) == 0:
        theta_by_pid = init_theta0_by_pid_from_df(
            df, id_col=id_col, price_col=price_col,
            input_dim=input_dim, hidden=hidden, device=device
        )
    else:
        if theta_prev_by_pid is None:
            raise ValueError("Para t>0 debes pasar theta_prev_by_pid = θ_{t-1} por jugador.")
        theta_by_pid = theta_prev_by_pid

    nets_by_pid = build_nets_by_pid(theta_by_pid, input_dim=input_dim, hidden=hidden, device=device)

    Beta, pids = beta_matrix_from_obs_batch(
      nets_by_pid,
      obs_batch_by_pid,
      device=device,
      enforce_beta_gt_phi=(enforce_beta_gt_phi and int(t) == 0),
      min_margin=min_margin,
      floor_mode=floor_mode,
  )


    return {"theta_by_pid": theta_by_pid, "nets_by_pid": nets_by_pid, "Beta": Beta, "pids": pids}


### PARTE 2.2 OBTENER EQUILIBRIO

In [14]:
import numpy as np
import pandas as pd
import ast

# ----------------------------
# 1) Parse helpers
# ----------------------------
def _parse_vec24_allow_fill(x, fill_value=np.nan) -> np.ndarray:
    """
    Convierte a (24,) aceptando lista/array o string '[...]'.
    Si es escalar, lo repite 24.
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.full(24, float(fill_value), dtype=float)

    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            x = ast.literal_eval(s)
        else:
            return np.full(24, float(s), dtype=float)

    if isinstance(x, (list, tuple, np.ndarray)):
        a = np.asarray(x, dtype=float).reshape(-1)
        if a.size == 24:
            return a
        if a.size == 1:
            return np.full(24, float(a[0]), dtype=float)
        raise ValueError(f"Se esperaba len 24 o escalar; got len={a.size}")

    return np.full(24, float(x), dtype=float)

# ----------------------------
# 2) Template (demanda + avail) desde df
# ----------------------------
def build_template_from_df(
    df: pd.DataFrame,
    *,
    id_col: str = "id_planta",
    avail_col: str = "available_q",
    demand_col: str = "daily_reman_demand",
) -> dict:
    """
    Output:
      template = {
        "I": array plant_ids (orden estable),
        "demanda": (24,),
        "_avail_mat": (I,24)
      }
    """
    for c in (id_col, avail_col, demand_col):
        if c not in df.columns:
            raise ValueError(f"Falta columna {c} en df")

    # demanda (24,)
    demand_raw = df[demand_col].dropna().iloc[-1]
    demand24 = _parse_vec24_allow_fill(demand_raw)

    # avail por planta (I,24) usando última fila por planta en orden estable
    last_av = (
        df.sort_index(kind="mergesort")
          .groupby(id_col, sort=False, as_index=False)
          .tail(1)[[id_col, avail_col]]
          .copy()
    )
    last_av["available_q_vec"] = last_av[avail_col].apply(_parse_vec24_allow_fill)

    plant_ids = [int(x) for x in last_av[id_col].to_numpy()]
    avail_mat = np.vstack(last_av["available_q_vec"].to_numpy()).astype(float)

    return {"I": np.array(plant_ids, dtype=int), "demanda": demand24.astype(float), "_avail_mat": avail_mat}

# ----------------------------
# 3) Construir juego_h desde Beta[h,:] y correr equilibrio
# ----------------------------
def compute_eq_for_batch_from_Beta(
    h_idx: int,
    *,
    template: dict,
    Beta: np.ndarray,                 # (H,I)
    pids: list,                       # len I
    compute_equilibrio_dispatch,      # tu función
    pricing: str = "uniform",
    unmet_price: float = np.nan,
) -> dict:
    plant_ids = [int(x) for x in np.asarray(template["I"]).reshape(-1)]
    avail_mat = np.asarray(template["_avail_mat"], dtype=float)
    demand24 = np.asarray(template["demanda"], dtype=float).reshape(24)

    Beta = np.asarray(Beta, dtype=float)
    if Beta.ndim != 2:
        raise ValueError(f"Beta debe ser 2D (H,I). Got shape={Beta.shape}")
    H, I = Beta.shape

    if not (0 <= h_idx < H):
        raise ValueError(f"h_idx fuera de rango. h_idx={h_idx}, H={H}")

    if len(pids) != I:
        raise ValueError(f"len(pids) debe ser I. len(pids)={len(pids)} vs I={I}")

    # map pid -> columna
    col = {int(pid): j for j, pid in enumerate(pids)}

    # chequeo: template plantas deben existir en pids
    missing = [pid for pid in plant_ids if pid not in col]
    if missing:
        raise KeyError(f"Estos pid del template no están en pids: {missing[:10]}")

    juego_h = {"I": np.array(plant_ids, dtype=int), "demanda": demand24.copy()}

    for r, pid in enumerate(plant_ids):
        j = col[pid]
        juego_h[pid] = {
            "beta": float(Beta[h_idx, j]),
            "available_q": avail_mat[r, :].copy(),
        }

    eq_h = compute_equilibrio_dispatch(juego_h, pricing=pricing, unmet_price=unmet_price)
    return eq_h

def compute_eq_over_batch_from_Beta(
    *,
    template: dict,
    Beta: np.ndarray,
    pids: list,
    compute_equilibrio_dispatch,
    pricing: str = "uniform",
    unmet_price: float = np.nan,
) -> list:
    Beta = np.asarray(Beta, dtype=float)
    H = int(Beta.shape[0])
    return [
        compute_eq_for_batch_from_Beta(
            h_idx=h,
            template=template,
            Beta=Beta,
            pids=pids,
            compute_equilibrio_dispatch=compute_equilibrio_dispatch,
            pricing=pricing,
            unmet_price=unmet_price,
        )
        for h in range(H)
    ]




In [15]:
import numpy as np

def compute_equilibrio_dispatch(juego: dict, pricing: str = "uniform", unmet_price="max_accepted"):
    """
    unmet_price:
      - número (float): usa ese precio si no hay dispatch
      - "max_accepted": si no hay dispatch, usa max bid positivo del sistema (si existe; si no, 0)
    """
    pricing = pricing.lower().strip()
    if pricing not in {"uniform", "pay_as_bid"}:
        raise ValueError("pricing must be 'uniform' or 'pay_as_bid'")

    # ids
    if "I" in juego:
        plant_ids = [int(x) for x in np.asarray(juego["I"]).reshape(-1)]
    else:
        plant_ids = [k for k in juego.keys() if isinstance(k, (int, np.integer))]

    if len(plant_ids) == 0:
        raise ValueError("No plant ids found in juego")

    demand24 = np.asarray(juego["demanda"], dtype=float).reshape(24)
    bids = np.array([float(juego[pid]["beta"]) for pid in plant_ids], dtype=float)  # (I,)
    avail_mat = np.vstack([np.asarray(juego[pid]["available_q"], dtype=float).reshape(24) for pid in plant_ids])  # (I,24)

    I = len(plant_ids)
    disp_mat = np.zeros((I, 24), dtype=float)
    price24 = np.zeros(24, dtype=float)  # nunca NaN

    # orden por bid (merit order), solo bids positivos
    active = bids > 0
    order = np.array([], dtype=int)
    if np.any(active):
        order = np.where(active)[0][np.argsort(bids[active], kind="mergesort")]

    # máximo bid positivo del sistema (para el caso “nadie despacha”)
    max_bid_pos = float(np.max(bids[active])) if np.any(active) else 0.0

    for h in range(24):
        remaining = float(max(demand24[h], 0.0))
        if remaining <= 0:
            price24[h] = 0.0
            continue

        av_h = np.maximum(avail_mat[:, h], 0.0)
        last_bid = None
        any_dispatch = False

        for i in order:
            if remaining <= 0:
                break
            take = min(av_h[i], remaining)
            if take > 0:
                disp_mat[i, h] = take
                remaining -= take
                any_dispatch = True
                last_bid = float(bids[i])

        # precio
        if any_dispatch:
            # “máximo aceptado” (marginal) incluso si quedó demanda sin cubrir
            price24[h] = float(last_bid)
        else:
            # nadie despachó -> evita NaN
            if unmet_price == "max_accepted":
                price24[h] = max_bid_pos
            else:
                price24[h] = float(unmet_price)

    # precio pagado por planta/hora
    if pricing == "uniform":
        precio_mat = np.where(disp_mat > 0.0, price24.reshape(1, 24), 0.0)
    else:
        precio_mat = np.where(disp_mat > 0.0, bids.reshape(I, 1), 0.0)

    eq = {
        "I": np.array(plant_ids, dtype=int),
        "demanda": demand24.copy(),
        "pricing": pricing,
        "price24": price24.copy(),
    }

    for idx, pid in enumerate(plant_ids):
        q = disp_mat[idx, :].copy()
        eq[pid] = {
            "beta": float(bids[idx]),
            "available_q": avail_mat[idx, :].copy(),
            "cantidades_despachadas": q,
            "precio24": precio_mat[idx, :].copy(),
            "sumaq": float(q.sum()),
        }
        if isinstance(juego.get(pid, {}), dict) and ("CodigoPlanta" in juego[pid]):
            eq[pid]["CodigoPlanta"] = juego[pid]["CodigoPlanta"]

    return eq


## PARTE 3: CALCULAR LAS JOINT UTILITIES

In [16]:
import numpy as np
from typing import Dict, Any, List, Tuple

def joint_utility_from_eq_list(
    eq_list: List[dict],                          # salida de compute_equilibrio_dispatch para cada h
    phi_batch_by_pid: Dict[Any, np.ndarray],      # phi_batch_by_pid[pid] = (H,24) o (H,)
    *,
    pricing: str = "uniform",
) -> Tuple[float, np.ndarray, Dict[Any, dict]]:
    """
    Calcula:
      U_total_by_batch[h] = sum_i u_i(h)
      U_total_mean        = mean_h U_total_by_batch[h]
      out_by_plant        = dict con utilidades por planta (mean y por batch)

    Convención de utilidad por planta i en escenario h:
      u_i(h) = sum_{t=1}^{24} q_{i,h,t} * (pago_{i,h,t} - phi_{i,h,t})

    donde:
      - q_{i,h,t} viene de eq_h[pid]["cantidades_despachadas"]
      - pago_{i,h,t} viene de eq_h[pid]["precio24"]
      - phi_{i,h,t} viene de phi_batch_by_pid[pid][h] (si escalar, se repite 24)

    Nota: el outcome (q, precios) ya viene dentro de eq_list y depende de β(o^h).
    """
    if len(eq_list) == 0:
        raise ValueError("eq_list vacío")

    H = len(eq_list)
    plant_ids = [int(x) for x in np.asarray(eq_list[0]["I"]).reshape(-1)]

    # utilidades por planta y por batch
    u_by_pid = {pid: np.zeros(H, dtype=float) for pid in plant_ids}

    for h_idx, eq_h in enumerate(eq_list):
        for pid in plant_ids:
            q24 = np.asarray(eq_h[pid]["cantidades_despachadas"], dtype=float).reshape(-1)
            if q24.size != 24:
                raise ValueError(f"pid={pid}: cantidades_despachadas debe tener 24, got {q24.size}")

            pay24 = np.asarray(eq_h[pid]["precio24"], dtype=float).reshape(-1)
            if pay24.size != 24:
                raise ValueError(f"pid={pid}: precio24 debe tener 24, got {pay24.size}")

            phi_h = np.asarray(phi_batch_by_pid[pid])
            if phi_h.ndim == 1:
                # escalar por h -> repetir 24
                c24 = np.full(24, float(phi_h[h_idx]), dtype=float)
            elif phi_h.ndim == 2:
                c24 = np.asarray(phi_h[h_idx, :], dtype=float).reshape(-1)
                if c24.size != 24:
                    raise ValueError(f"pid={pid}: phi_h[h] debe tener 24, got {c24.size}")
            else:
                raise ValueError(f"pid={pid}: phi_batch_by_pid[pid] shape no soportado {phi_h.shape}")

            u_by_pid[pid][h_idx] = float((q24 * (pay24 - c24)).sum())

    # utilidad conjunta por batch: sum_i u_i(h)
    U_total_by_batch = np.zeros(H, dtype=float)
    for h_idx in range(H):
        U_total_by_batch[h_idx] = sum(u_by_pid[pid][h_idx] for pid in plant_ids)

    U_total_mean = float(U_total_by_batch.mean())

    # salida detallada por planta
    out_by_plant: Dict[Any, dict] = {}
    for pid in plant_ids:
        out_by_plant[pid] = {
            "u_mean": float(u_by_pid[pid].mean()),
            "u_by_batch": u_by_pid[pid].copy(),
        }

    out_by_plant["_meta"] = {"pricing": str(pricing), "H": int(H), "plant_ids": plant_ids}

    return U_total_mean, U_total_by_batch, out_by_plant


## PARTE 4: P PERTURBACIONES OF THE POLICY AND OBTAIN GRADIENT

In [17]:
import numpy as np
import torch
import torch.nn.functional as F
from typing import Dict, Any, List, Tuple

TensorDict = Dict[str, torch.Tensor]


# ============================================================
# 2.1) Dispatch: ahora acepta beta escalar O beta vector (24,)
#      (mismo nombre: compute_equilibrio_dispatch)
# ============================================================
def compute_equilibrio_dispatch(juego: dict, pricing: str = "uniform", unmet_price=np.nan):
    """
    Input juego:
      juego["I"] (opcional) : array/list de ids
      juego["demanda"]      : array (24,)
      juego[pid]["beta"]    : float (bid escalar)  o  array-like (24,) (bid horario)
      juego[pid]["available_q"] : array (24,)

    Output eq:
      eq["price24"] : (24,) precio marginal por hora
      eq[pid]["beta"] : (24,)  (si bid era escalar, se expande)
      eq[pid]["cantidades_despachadas"] : (24,)
      eq[pid]["precio24"] : (24,)
    """
    pricing = pricing.lower().strip()
    if pricing not in {"uniform", "pay_as_bid"}:
        raise ValueError("pricing must be 'uniform' or 'pay_as_bid'")

    # ids
    if "I" in juego:
        plant_ids = [int(x) for x in np.asarray(juego["I"]).reshape(-1)]
    else:
        plant_ids = [k for k in juego.keys() if isinstance(k, (int, np.integer))]

    if len(plant_ids) == 0:
        raise ValueError("No plant ids found in juego")

    demand24 = np.asarray(juego["demanda"], dtype=float).reshape(24)

    # bids24_mat: (I,24) (si beta era escalar, se expande)
    bids24 = []
    for pid in plant_ids:
        b = juego[pid]["beta"]
        if np.isscalar(b):
            bids24.append(np.full(24, float(b), dtype=float))
        else:
            bb = np.asarray(b, dtype=float).reshape(-1)
            if bb.size != 24:
                raise ValueError(f"pid={pid}: beta vector debe tener 24, got {bb.size}")
            bids24.append(bb)
    bids24_mat = np.vstack(bids24)  # (I,24)

    avail_mat = np.vstack([
        np.asarray(juego[pid]["available_q"], dtype=float).reshape(24)
        for pid in plant_ids
    ])  # (I,24)

    I = len(plant_ids)
    disp_mat = np.zeros((I, 24), dtype=float)
    price24 = np.full(24, float(unmet_price), dtype=float)

    # dispatch por hora (merit order por hora si bids horarios)
    for h in range(24):
        remaining = float(max(demand24[h], 0.0))
        if remaining <= 0:
            price24[h] = 0.0
            continue

        bids_h = bids24_mat[:, h]
        av_h = np.maximum(avail_mat[:, h], 0.0)

        active = bids_h > 0
        if not np.any(active):
            price24[h] = float(unmet_price)
            continue

        order = np.where(active)[0][np.argsort(bids_h[active], kind="mergesort")]

        last_bid = None
        any_dispatch = False
        for i in order:
            if remaining <= 0:
                break
            take = min(av_h[i], remaining)
            if take > 0:
                disp_mat[i, h] = take
                remaining -= take
                any_dispatch = True
                last_bid = float(bids_h[i])

        price24[h] = float(last_bid) if any_dispatch else float(unmet_price)

    # pagos por planta/hora
    if pricing == "uniform":
        precio_mat = np.where(disp_mat > 0.0, price24.reshape(1, 24), 0.0)
    else:
        precio_mat = np.where(disp_mat > 0.0, bids24_mat, 0.0)

    eq = {
        "I": np.array(plant_ids, dtype=int),
        "demanda": demand24.copy(),
        "pricing": pricing,
        "price24": price24.copy(),
    }

    for idx, pid in enumerate(plant_ids):
        q = disp_mat[idx, :].copy()
        eq[pid] = {
            "beta": bids24_mat[idx, :].copy(),  # siempre (24,)
            "available_q": avail_mat[idx, :].copy(),
            "cantidades_despachadas": q,
            "precio24": precio_mat[idx, :].copy(),
            "sumaq": float(q.sum()),
        }

    return eq


# ============================================================
# 2.2) Equilibrio por batch: ahora acepta Beta (H,I) o (H,I,24)
#      (mismo nombre: compute_eq_over_batch_from_Beta)
# ============================================================
def compute_eq_over_batch_from_Beta(
    *,
    template: dict,                  # {"I": (I,), "demanda": (24,), "_avail_mat": (I,24)}
    Beta: np.ndarray,                # (H,I) o (H,I,24)
    pids: List[int],                 # orden columnas de Beta
    compute_equilibrio_dispatch,     # esta misma función (o compatible)
    pricing: str = "uniform",
    unmet_price: float = np.nan,
) -> List[dict]:
    Beta = np.asarray(Beta, dtype=float)

    if Beta.ndim == 2:
        H, I = Beta.shape
        is_hourly = False
    elif Beta.ndim == 3 and Beta.shape[2] == 24:
        H, I, _ = Beta.shape
        is_hourly = True
    else:
        raise ValueError(f"Beta debe ser (H,I) o (H,I,24). Got {Beta.shape}")

    plant_ids = [int(x) for x in np.asarray(template["I"]).reshape(-1)]
    avail_mat = np.asarray(template["_avail_mat"], dtype=float)
    demand24 = np.asarray(template["demanda"], dtype=float).reshape(24)

    col = {int(pid): j for j, pid in enumerate(pids)}
    missing = [pid for pid in plant_ids if pid not in col]
    if missing:
        raise KeyError(f"template['I'] contiene pids que no están en pids: {missing[:10]}")

    eq_list = []
    for h_idx in range(H):
        juego_h = {"I": np.array(plant_ids, dtype=int), "demanda": demand24.copy()}
        for r, pid in enumerate(plant_ids):
            j = col[int(pid)]
            beta_val = Beta[h_idx, j] if not is_hourly else Beta[h_idx, j, :].copy()
            juego_h[int(pid)] = {
                "beta": beta_val,
                "available_q": avail_mat[r, :].copy(),
            }
        eq_h = compute_equilibrio_dispatch(juego_h, pricing=pricing, unmet_price=unmet_price)
        eq_list.append(eq_h)

    return eq_list


# ============================================================
# 3) Recalcular SOLO la columna beta_i^h para un theta dado
#    MISMO NOMBRE: eval_policy_column_over_batch
#    - si obs_batch es (H,)    -> retorna (H,)
#    - si obs_batch es (H,24)  -> retorna (H,24) y garantiza beta>=phi elemento a elemento
# ============================================================
import numpy as np
import torch
import torch.nn.functional as F

# ============================================================
# (1) Eval beta_i^h para UNA planta (SIEMPRE devuelve (H,))
#     Restricción: beta^h > phi^h (si phi es escalar)
#     Si obs es (H,24), impone beta^h > max(phi_h)  -> > cada hora
# ============================================================
def eval_policy_column_over_batch(
    *,
    net,
    theta: TensorDict,
    obs_batch: np.ndarray,        # (H,) o (H,24) o (H,d)
    device: str = "auto",
    min_bid: float = 1e-6,
    floor_mode: str = "max",      # si obs es 2D: max/mean para piso phi
    reduce_mode: str = "max",     # si red devuelve (H,24): max/mean para colapsar
) -> np.ndarray:
    from torch.func import functional_call

    net = net.to(device)
    net.eval()

    obs = np.asarray(obs_batch, dtype=np.float32)

    if obs.ndim == 1:
        X = torch.tensor(obs.reshape(-1, 1), dtype=torch.float32, device=device)  # (H,1)
        floor = obs.astype(float)                                                # (H,)
    elif obs.ndim == 2:
        X = torch.tensor(obs, dtype=torch.float32, device=device)                # (H,d)
        if floor_mode == "max":
            floor = obs.max(axis=1).astype(float)                                # (H,)
        elif floor_mode == "mean":
            floor = obs.mean(axis=1).astype(float)
        else:
            raise ValueError("floor_mode must be 'max' or 'mean'")
    else:
        raise ValueError(f"obs_batch shape no soportado: {obs.shape}")

    theta_dev = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in theta.items()}

    with torch.no_grad():
        y = functional_call(net, theta_dev, (X,))  # (H,1) o (H,24) o (H,)

        # colapsar salida a escalar por h
        if y.ndim == 2 and y.shape[1] > 1:
            if reduce_mode == "max":
                raw = y.max(dim=1).values
            elif reduce_mode == "mean":
                raw = y.mean(dim=1)
            else:
                raise ValueError("reduce_mode must be 'max' or 'mean'")
        else:
            raw = y.reshape(-1)

        inc = F.softplus(raw).detach().cpu().numpy().astype(float)               # >=0
        beta = floor + inc + float(min_bid)                                      # (H,)

    return beta





# ============================================================
# 4) ES para una planta (MISMO NOMBRE)
#    - ahora soporta Beta_base (H,I) o (H,I,24)
#    - y soporta beta_col_p (H,) o (H,24) según obs_batch
# ============================================================
import numpy as np
import torch
from typing import Dict, Any, List

TensorDict = Dict[str, torch.Tensor]

# FIX: tu versión de perturb_single_agent_es_individual no tenía el parámetro reduce_mode.
# Añádelo al signature y pásalo a eval_policy_column_over_batch.

def perturb_single_agent_es_individual(
    *,
    planta: int,
    theta_by_pid,
    net_by_pid,
    obs_batch_by_pid,
    phi_batch_by_pid,
    template,
    Beta_tminus1,
    pids,
    u_mean_base_planta: float,
    compute_equilibrio_dispatch,
    joint_utility_from_eq_list,
    pricing="uniform",
    unmet_price=np.nan,
    P=64,
    sigma=1e-3,
    seed=123,
    device="auto",
    min_bid=1e-6,
    floor_mode="max",
    reduce_mode="max",   # <-- AÑADIR
):
    import numpy as np
    import torch

    Beta_base = np.asarray(Beta_tminus1, dtype=float)
    H, I = Beta_base.shape

    planta = int(planta)
    col = {int(pid): j for j, pid in enumerate(pids)}
    if planta not in col:
        raise KeyError(f"planta={planta} no está en pids")
    j_planta = col[planta]

    theta0 = theta_by_pid[planta]

    grad = {
        k: torch.zeros_like(v)
        for k, v in theta0.items()
        if torch.is_tensor(v) and torch.is_floating_point(v)
    }

    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    fitness_list = np.zeros(int(P), dtype=float)
    umean_list = np.zeros(int(P), dtype=float)

    for p in range(int(P)):
        # eps_p ~ N(0, sigma^2 I)  (perturbación directa)
        eps = {}
        theta_p = {}
        for k, v in theta0.items():
            if torch.is_tensor(v) and torch.is_floating_point(v):
                e = torch.randn_like(v) * float(sigma)
                eps[k] = e
                theta_p[k] = v + e
            else:
                theta_p[k] = v

        beta_col_p = eval_policy_column_over_batch(
            net=net_by_pid[planta],
            theta=theta_p,
            obs_batch=obs_batch_by_pid[planta],
            device=device,
            min_bid=min_bid,
            floor_mode=floor_mode,
            reduce_mode=reduce_mode,   # <-- PASAR
        )  # (H,)

        Beta_p = Beta_base.copy()
        Beta_p[:, j_planta] = beta_col_p

        eq_list_p = compute_eq_over_batch_from_Beta(
            template=template,
            Beta=Beta_p,
            pids=pids,
            compute_equilibrio_dispatch=compute_equilibrio_dispatch,
            pricing=pricing,
            unmet_price=unmet_price,
        )

        _Umean, _Ubyh, out_by_plant_p = joint_utility_from_eq_list(
            eq_list_p,
            phi_batch_by_pid=phi_batch_by_pid,
            pricing=pricing,
        )

        u_mean_p = float(out_by_plant_p[planta]["u_mean"])
        fitness = float(u_mean_p - float(u_mean_base_planta))

        fitness_list[p] = fitness
        umean_list[p] = u_mean_p

        for k in grad.keys():
            grad[k].add_(fitness * eps[k])

    # ∇ = 1/(sigma^2 P) Σ fitness * eps
    scale = 1.0 / (float(sigma) * float(sigma) * float(P))
    for k in grad.keys():
        grad[k].mul_(scale)

    return {
        "planta": int(planta),
        "col_index": int(j_planta),
        "grad_theta": grad,
        "fitness": fitness_list,
        "u_mean_pert": umean_list,
        "u_mean_base": float(u_mean_base_planta),
        "sigma": float(sigma),
        "P": int(P)
    }


# ============================================================
# 5) Runner (MISMO NOMBRE)
#    - baseline por planta: usa out_by_plant_base[pid]["u_mean"]
# ============================================================
def run_es_for_all_plants(
    *,
    plant_ids: List[int],
    bids: Dict[int, Dict[str, Any]],      # bids[pid]["theta"] existe
    net_by_pid: Dict[int, Any],
    obs_batch_by_pid: Dict[int, np.ndarray],
    template: dict,
    Beta_tminus1: np.ndarray,
    pids: List[int],
    compute_equilibrio_dispatch,
    pricing: str = "uniform",
    unmet_price: float = np.nan,
    P: int = 64,
    sigma: float = 1e-3,
    seed: int = 123,
    device: str = "auto",
) -> Dict[str, Any]:
    # baseline
    eq_list_base = compute_eq_over_batch_from_Beta(
        template=template,
        Beta=Beta_tminus1,
        pids=pids,
        compute_equilibrio_dispatch=compute_equilibrio_dispatch,
        pricing=pricing,
        unmet_price=unmet_price,
    )
    U_mean_base, U_by_h_base, out_by_plant_base = joint_utility_from_eq_list(
        eq_list_base,
        phi_batch_by_pid=obs_batch_by_pid,
        pricing=pricing,
    )

    theta_by_pid = {int(pid): bids[int(pid)]["theta"] for pid in plant_ids}

    results = {}
    for pid in plant_ids:
        pid = int(pid)
        u0_pid = float(out_by_plant_base[pid]["u_mean"])
        results[pid] = perturb_single_agent_es_individual(
            planta=pid,
            theta_by_pid=theta_by_pid,
            net_by_pid=net_by_pid,
            obs_batch_by_pid=obs_batch_by_pid,
            phi_batch_by_pid=obs_batch_by_pid,
            template=template,
            Beta_base=Beta_tminus1,
            pids=pids,
            u_mean_base_planta=u0_pid,  # <- baseline individual correcto
            compute_equilibrio_dispatch=compute_equilibrio_dispatch,
            pricing=pricing,
            unmet_price=unmet_price,
            P=P,
            sigma=sigma,
            seed=seed,
            device=device,
        )

    return {
        "U_mean_base": U_mean_base,
        "U_by_h_base": U_by_h_base,
        "out_by_plant_base": out_by_plant_base,
        "es_results": results,
    }


## PARTE 5: ACTUALIZAR PARAMETROS DE LA RED

In [18]:
import numpy as np
import torch
import torch.nn.functional as F
from typing import Dict, Any, List, Tuple

TensorDict = Dict[str, torch.Tensor]


# ============================================================
# (2) NPGA step para UN agente i
#     θ_i^t = θ_i^{t-1} + η * grad_ES_i
#     β_i^t(o^h) = π_i(o^h; θ_i^t)  (H,)
# ============================================================
def npga_step_single_agent(
    *,
    planta_id: int,
    theta_by_pid: Dict[int, TensorDict],      # θ_{t-1}
    net_by_pid: Dict[int, Any],               # PolicyNet por pid (arquitectura)
    obs_batch_by_pid: Dict[int, np.ndarray],  # o_i^h (phi)
    Beta_tminus1: np.ndarray,                 # (H,I) bids baseline
    pids: List[int],                          # orden columnas
    res_i: dict,                              # salida ES: res_i["grad_theta"], res_i["planta"]
    eta: float,
    device: str = "auto",
    min_margin: float = 1e-6,
    floor_mode: str = "max",
    reduce_mode: str = "max",
) -> Tuple[Dict[int, TensorDict], np.ndarray, float]:
    pid = int(planta_id)

    # columna j del agente en Beta
    col = {int(x): j for j, x in enumerate(pids)}
    if pid not in col:
        raise KeyError(f"planta_id={pid} no está en pids")
    j = col[pid]

    # grad ES del agente
    if int(res_i.get("planta", pid)) != pid:
        raise ValueError("res_i no corresponde a planta_id")
    grad = res_i["grad_theta"]

    # 1) θ update
    theta_prev = theta_by_pid[pid]
    theta_new: TensorDict = {}
    for k, v in theta_prev.items():
        if torch.is_tensor(v) and torch.is_floating_point(v) and (k in grad):
            theta_new[k] = (v + float(eta) * grad[k].to(v.device)).detach().clone()
        elif torch.is_tensor(v):
            theta_new[k] = v.detach().clone()
        else:
            theta_new[k] = v
    theta_by_pid[pid] = theta_new

    # 2) β_i^t(o^h): SIEMPRE (H,)
    beta_col_new = eval_policy_column_over_batch(
        net=net_by_pid[pid],
        theta=theta_new,
        obs_batch=obs_batch_by_pid[pid],
        device=device,
        min_bid=min_margin,
        floor_mode=floor_mode,
        reduce_mode=reduce_mode,
    )
    if beta_col_new.ndim != 1:
        raise ValueError(f"beta_col_new debe ser (H,), got {beta_col_new.shape}")

    # 3) Beta_t (solo cambia columna del agente)
    Beta_t = np.asarray(Beta_tminus1, dtype=float).copy()
    if Beta_t.ndim != 2:
        raise ValueError(f"Beta_tminus1 debe ser (H,I). Got {Beta_t.shape}")
    Beta_t[:, j] = beta_col_new

    # 4) beta "por planta" (escalar): promedio sobre escenarios h
    beta_t_scalar = float(np.mean(beta_col_new))

    return theta_by_pid, Beta_t, beta_t_scalar


## PARTE 6: REPETIR PARTE 4 Y 5 PARA CADA I

In [19]:
import numpy as np
import torch
from typing import Dict, Any, List, Tuple

TensorDict = Dict[str, torch.Tensor]

# ============================================================
# 1) ES "simultáneo": para cada i, oponentes fijos en t-1
#    - baseline usa Beta_{t-1}(o^h) de TODOS con θ_{t-1}
#    - para i: reemplazas SOLO su columna por β_i^p(o^h) y recomputas eq/utilidad
# ============================================================
def run_es_for_all_plants(
    *,
    plant_ids: List[int],
    theta_by_pid: Dict[int, TensorDict],          # θ_{t-1}
    net_by_pid: Dict[int, Any],                   # PolicyNet por pid
    obs_batch_by_pid: Dict[int, np.ndarray],      # o_i^h (phi)
    template: dict,
    compute_equilibrio_dispatch,
    joint_utility_from_eq_list,
    pricing: str = "uniform",
    unmet_price: float = np.nan,
    P: int = 64,
    sigma: float = 1e-3,
    seed: int = 123,
    device: str = "auto",
    min_margin: float = 1e-6,
    floor_mode: str = "max",
) -> dict:
    """
    Devuelve:
      {
        "Beta_base": Beta_{t-1}(o^h),
        "pids": orden columnas,
        "out_by_plant_base": utilidades baseline,
        "es_results": {pid: res_i}  (grad ES por planta, con oponentes fijos)
      }
    """
    pids = [int(pid) for pid in plant_ids]
    H = int(np.asarray(obs_batch_by_pid[pids[0]]).shape[0])
    I = len(pids)

    # ---- construir Beta_base (H,I) con restricción β>φ usando eval_policy_column_over_batch ----
    Beta_base = np.zeros((H, I), dtype=float)
    for j, pid in enumerate(pids):
        Beta_base[:, j] = eval_policy_column_over_batch(
            net=net_by_pid[pid],
            theta=theta_by_pid[pid],
            obs_batch=obs_batch_by_pid[pid],
            device=device,
            min_bid=min_margin,
            floor_mode=floor_mode,
        )

    # ---- baseline: equilibrios + utilidades ----
    eq_list_base = compute_eq_over_batch_from_Beta(
        template=template,
        Beta=Beta_base,
        pids=pids,
        compute_equilibrio_dispatch=compute_equilibrio_dispatch,
        pricing=pricing,
        unmet_price=unmet_price,
    )
    U_mean_base, U_by_h_base, out_by_plant_base = joint_utility_from_eq_list(
        eq_list_base,
        phi_batch_by_pid=obs_batch_by_pid,   # v=phi
        pricing=pricing,
    )

    # ---- ES por planta, oponentes fijos en Beta_base ----
    es_results: Dict[int, dict] = {}
    for pid in pids:
        u_mean_base_i = float(out_by_plant_base[pid]["u_mean"])

        res_i = perturb_single_agent_es_individual(
            planta=pid,
            theta_by_pid=theta_by_pid,
            net_by_pid=net_by_pid,
            obs_batch_by_pid=obs_batch_by_pid,
            phi_batch_by_pid=obs_batch_by_pid,
            template=template,
            Beta_tminus1=Beta_base,                  # <- fijo para oponentes
            pids=pids,
            u_mean_base_planta=u_mean_base_i,
            compute_equilibrio_dispatch=compute_equilibrio_dispatch,
            joint_utility_from_eq_list=joint_utility_from_eq_list,
            pricing=pricing,
            unmet_price=unmet_price,
            P=P,
            sigma=sigma,
            seed=seed,
            device=device,
            min_bid=min_margin,
            floor_mode=floor_mode,
        )
        es_results[pid] = res_i

    return {
        "Beta_base": Beta_base,
        "pids": pids,
        "U_mean_base": U_mean_base,
        "U_by_h_base": U_by_h_base,
        "out_by_plant_base": out_by_plant_base,
        "es_results": es_results,
    }


# ============================================================
# 2) Update "simultáneo": aplicas TODOS los gradientes a θ_{t-1}
#    y luego recomputas Beta_t(o^h) con esos θ_t
# ============================================================
def apply_es_updates_for_all_plants(
    *,
    plant_ids: List[int],
    theta_by_pid: Dict[int, TensorDict],          # θ_{t-1}
    es_results: Dict[int, dict],                  # output["es_results"]
    eta: float,
    net_by_pid: Dict[int, Any],                   # nets (arquitectura)
    obs_batch_by_pid: Dict[int, np.ndarray],      # o_i^h
    template: dict,
    compute_equilibrio_dispatch,
    joint_utility_from_eq_list,
    pricing: str = "uniform",
    unmet_price: float = np.nan,
    device: str = "auto",
    min_margin: float = 1e-6,
    floor_mode: str = "max",
) -> dict:
    """
    Devuelve:
      {
        "theta_by_pid": θ_t,
        "Beta_t": β_t(o^h),
        "pids": orden columnas,
        "eq_list_t": (opcional) equilibrios con Beta_t,
        "out_by_plant_t": utilidades con θ_t
      }
    """
    pids = [int(pid) for pid in plant_ids]
    col = {pid: j for j, pid in enumerate(pids)}

    # ---- θ_t = θ_{t-1} + η * grad_ES (para TODOS, sin mezclar) ----
    theta_new_by_pid: Dict[int, TensorDict] = {pid: theta_by_pid[pid] for pid in pids}

    for pid in pids:
        grad = es_results[pid]["grad_theta"]
        theta_prev = theta_by_pid[pid]
        theta_new: TensorDict = {}

        for k, v in theta_prev.items():
            if torch.is_tensor(v) and torch.is_floating_point(v) and (k in grad):
                theta_new[k] = (v + float(eta) * grad[k].to(v.device)).detach().clone()
            elif torch.is_tensor(v):
                theta_new[k] = v.detach().clone()
            else:
                theta_new[k] = v

        theta_new_by_pid[pid] = theta_new

    # ---- recomputar Beta_t con θ_t (restricción β>φ) ----
    H = int(np.asarray(obs_batch_by_pid[pids[0]]).shape[0])
    I = len(pids)
    Beta_t = np.zeros((H, I), dtype=float)

    for j, pid in enumerate(pids):
        Beta_t[:, j] = eval_policy_column_over_batch(
            net=net_by_pid[pid],
            theta=theta_new_by_pid[pid],
            obs_batch=obs_batch_by_pid[pid],
            device=device,
            min_bid=min_margin,
            floor_mode=floor_mode,
        )

    # (opcional) evaluar eq/utilidades en θ_t
    eq_list_t = compute_eq_over_batch_from_Beta(
        template=template,
        Beta=Beta_t,
        pids=pids,
        compute_equilibrio_dispatch=compute_equilibrio_dispatch,
        pricing=pricing,
        unmet_price=unmet_price,
    )
    U_mean_t, U_by_h_t, out_by_plant_t = joint_utility_from_eq_list(
        eq_list_t,
        phi_batch_by_pid=obs_batch_by_pid,
        pricing=pricing,
    )

    return {
        "theta_by_pid": theta_new_by_pid,
        "Beta_t": Beta_t,
        "pids": pids,
        "eq_list_t": eq_list_t,
        "U_mean_t": U_mean_t,
        "U_by_h_t": U_by_h_t,
        "out_by_plant_t": out_by_plant_t,
    }




## PARTE 7: REPETIR PARTE 2, 3  Y 6 PARA CADA t


In [20]:
import numpy as np
import torch
from typing import Dict, Any, List

TensorDict = Dict[str, torch.Tensor]


def run_NPGA_all(
    *,
    t: int,
    plant_ids: List[int],
    df,                                       # solo si t==0 para inicializar θ0
    obs_batch_by_pid: Dict[int, np.ndarray],  # o_i^h (phi)
    template: dict,
    compute_equilibrio_dispatch,
    joint_utility_from_eq_list,
    pricing: str = "uniform",
    unmet_price: float = np.nan,
    # NPGA hyperparams
    eta: float = 1e-4,
    P: int = 64,
    sigma: float = 1e-3,
    seed: int = 123,
    device: str = "auto",
    # policy arch
    input_dim: int = 1,
    hidden=(32, 64, 128, 128, 64, 32),
    # beta constraint beta > phi
    min_margin: float = 1e-6,
    floor_mode: str = "max",
    reduce_mode: str = "max",
    # state (required if t>0)
    theta_prev_by_pid: Dict[int, TensorDict] = None,
) -> dict:
    """
    Hace TODO el periodo t:

    1) Construye θ_{t-1}:
       - si t==0: inicializa θ0 desde df (política constante por planta)
       - si t>0: usa theta_prev_by_pid provisto

    2) Construye nets_by_pid (una PolicyNet por planta) y calcula Beta_{t-1}(o^h) con restricción beta>phi

    3) Baseline: eq_list_base + utilidades baseline

    4) Para cada planta i:
       - P perturbaciones θ_i^p = θ_i + ε_p,  ε_p ~ N(0, sigma^2 I)
       - recalcula SOLO su columna beta_i^p(o^h), mantiene oponentes fijos (t-1)
       - recomputa eq/utilidad, fitness = u_mean_i^p - u_mean_i_base
       - grad_ES_i = (1/(sigma^2 P)) Σ_p fitness_p * ε_p

    5) Update simultáneo:
       θ_i^t = θ_i^{t-1} + eta * grad_ES_i   para todos i

    6) Recalcula Beta_t(o^h) desde θ_t (restricción beta>phi)
       y (opcional) eq/utilidades en t

    Devuelve todo para encadenar al siguiente t.
    """
    # ---------- θ_{t-1} ----------
    pids = [int(x) for x in plant_ids]
    if int(t) == 0:
        theta_prev_by_pid = init_theta0_by_pid_from_df(
            df,
            id_col="id_planta",
            price_col="precio_d",
            input_dim=input_dim,
            hidden=hidden,
            device=device,
        )
        theta_prev_by_pid = {int(pid): theta_prev_by_pid[int(pid)] for pid in pids}
    else:
        if theta_prev_by_pid is None:
            raise ValueError("Para t>0 debes pasar theta_prev_by_pid = θ_{t-1}.")
        theta_prev_by_pid = {int(pid): theta_prev_by_pid[int(pid)] for pid in pids}

    # ---------- nets_{t-1} ----------
    net_by_pid = build_nets_by_pid(theta_prev_by_pid, input_dim=input_dim, hidden=hidden, device=device)

    # ---------- Beta_{t-1}(o^h) ----------
    H = int(np.asarray(obs_batch_by_pid[pids[0]]).shape[0])
    I = len(pids)
    Beta_base = np.zeros((H, I), dtype=float)
    for j, pid in enumerate(pids):
        Beta_base[:, j] = eval_policy_column_over_batch(
            net=net_by_pid[pid],
            theta=theta_prev_by_pid[pid],
            obs_batch=obs_batch_by_pid[pid],
            device=device,
            min_bid=min_margin,
            floor_mode=floor_mode,
            reduce_mode=reduce_mode,
        )

    # ---------- baseline eq/util ----------
    eq_list_base = compute_eq_over_batch_from_Beta(
        template=template,
        Beta=Beta_base,
        pids=pids,
        compute_equilibrio_dispatch=compute_equilibrio_dispatch,
        pricing=pricing,
        unmet_price=unmet_price,
    )
    U_mean_base, U_by_h_base, out_by_plant_base = joint_utility_from_eq_list(
        eq_list_base,
        phi_batch_by_pid=obs_batch_by_pid,
        pricing=pricing,
    )

    # ---------- ES por planta (simultáneo: oponentes fijos en Beta_base) ----------
    es_results: Dict[int, dict] = {}
    for pid in pids:
        u_mean_base_i = float(out_by_plant_base[pid]["u_mean"])
        es_results[pid] = perturb_single_agent_es_individual(
            planta=pid,
            theta_by_pid=theta_prev_by_pid,
            net_by_pid=net_by_pid,
            obs_batch_by_pid=obs_batch_by_pid,
            phi_batch_by_pid=obs_batch_by_pid,
            template=template,
            Beta_tminus1=Beta_base,
            pids=pids,
            u_mean_base_planta=u_mean_base_i,
            compute_equilibrio_dispatch=compute_equilibrio_dispatch,
            joint_utility_from_eq_list=joint_utility_from_eq_list,
            pricing=pricing,
            unmet_price=unmet_price,
            P=P,
            sigma=sigma,
            seed=seed,
            device=device,
            min_bid=min_margin,
            floor_mode=floor_mode,
            reduce_mode=reduce_mode,
        )

    # ---------- Update simultáneo θ_t ----------
    theta_t_by_pid: Dict[int, TensorDict] = {}
    for pid in pids:
        grad = es_results[pid]["grad_theta"]
        theta_prev = theta_prev_by_pid[pid]
        theta_new: TensorDict = {}
        for k, v in theta_prev.items():
            if torch.is_tensor(v) and torch.is_floating_point(v) and (k in grad):
                theta_new[k] = (v + float(eta) * grad[k].to(v.device)).detach().clone()
            elif torch.is_tensor(v):
                theta_new[k] = v.detach().clone()
            else:
                theta_new[k] = v
        theta_t_by_pid[pid] = theta_new

    # ---------- Recalcular Beta_t(o^h) ----------
    net_t_by_pid = build_nets_by_pid(theta_t_by_pid, input_dim=input_dim, hidden=hidden, device=device)

    Beta_t = np.zeros((H, I), dtype=float)
    for j, pid in enumerate(pids):
        Beta_t[:, j] = eval_policy_column_over_batch(
            net=net_t_by_pid[pid],
            theta=theta_t_by_pid[pid],
            obs_batch=obs_batch_by_pid[pid],
            device=device,
            min_bid=min_margin,
            floor_mode=floor_mode,
            reduce_mode=reduce_mode,
        )

    # ---------- (opcional) eq/util en t ----------
    eq_list_t = compute_eq_over_batch_from_Beta(
        template=template,
        Beta=Beta_t,
        pids=pids,
        compute_equilibrio_dispatch=compute_equilibrio_dispatch,
        pricing=pricing,
        unmet_price=unmet_price,
    )
    U_mean_t, U_by_h_t, out_by_plant_t = joint_utility_from_eq_list(
        eq_list_t,
        phi_batch_by_pid=obs_batch_by_pid,
        pricing=pricing,
    )

    # beta escalar por planta (promedio sobre h)
    beta_t_scalar_by_pid = {pid: float(np.mean(Beta_t[:, j])) for j, pid in enumerate(pids)}

    return {
        "t": int(t),
        "pids": pids,

        # baseline (t-1)
        "theta_prev_by_pid": theta_prev_by_pid,
        "net_prev_by_pid": net_by_pid,
        "Beta_tminus1": Beta_base,
        "eq_list_base": eq_list_base,
        "U_mean_base": U_mean_base,
        "U_by_h_base": U_by_h_base,
        "out_by_plant_base": out_by_plant_base,

        # ES grads
        "es_results": es_results,

        # updated (t)
        "theta_by_pid": theta_t_by_pid,
        "net_by_pid": net_t_by_pid,
        "Beta_t": Beta_t,
        "beta_t_scalar_by_pid": beta_t_scalar_by_pid,
        "eq_list_t": eq_list_t,
        "U_mean_t": U_mean_t,
        "U_by_h_t": U_by_h_t,
        "out_by_plant_t": out_by_plant_t,
    }



In [21]:
import numpy as np
import torch
from typing import Dict, Any, List
from tqdm.auto import trange

TensorDict = Dict[str, torch.Tensor]


def run_NPGA(
    *,
    T: int,                                # número de iteraciones (t = 0,...,T-1)
    H: int,                                # batch size (número de escenarios h)
    plant_ids: List[int],
    df,
    # sampling (debe devolver obs_batch_by_pid: Dict[pid] -> (H,) o (H,24) )
    sample_obs_batch_fn,                   # ej: sample_day_batches_phi
    sample_obs_batch_kwargs: dict = None,  # kwargs extra para tu sampler (day_col, phi_col, etc.)
    # mechanism + utility
    build_template_fn,                     # ej: build_template_from_df
    compute_equilibrio_dispatch,
    joint_utility_from_eq_list,
    pricing: str = "uniform",
    unmet_price: float = np.nan,
    # NPGA hyperparams
    eta: float = 1e-4,
    P: int = 64,
    sigma: float = 1e-3,
    seed: int = 123,
    device: str = "auto",
    # policy arch
    input_dim: int = 1,
    hidden=(32, 64, 128, 128, 64, 32),
    # constraint beta > phi
    min_margin: float = 1e-6,
    floor_mode: str = "max",
    reduce_mode: str = "max",
    # initial state (opcional si quieres arrancar desde θ ya entrenado)
    theta_init_by_pid: Dict[int, TensorDict] = None,
    store_history: bool = True,
) -> dict:
    """
    Ejecuta el loop completo:
      for t=0..T-1:
        sample obs_batch_by_pid (H escenarios)
        Beta_{t}(o^h) desde θ_t
        baseline eq/util
        ES grads por planta (oponentes fijos en t)
        θ_{t+1} = θ_t + eta * grad_ES (simultáneo)
    Devuelve θ final y (opcional) historia.
    """

    pids = [int(x) for x in plant_ids]
    sample_obs_batch_kwargs = sample_obs_batch_kwargs or {}

    # template fijo (demanda + avail)
    template = build_template_fn(df)

    # θ_0
    if theta_init_by_pid is None:
        theta_by_pid, _ = init_theta0_by_pid_from_df(
            df,
            id_col="id_planta",
            price_col="precio_d",
            input_dim=input_dim,
            hidden=hidden,
            device=device,
        )
        theta_by_pid = {int(pid): theta_by_pid[int(pid)] for pid in pids}
    else:
        theta_by_pid = {int(pid): theta_init_by_pid[int(pid)] for pid in pids}

    history = {
        "U_mean": [],
        "beta_scalar_by_pid": [],
        "out_by_plant": [],
        "es_results": [],
        "Beta": [],
    } if store_history else None

    base_seed = int(seed)

    for t in trange(int(T), desc="NPGA", unit="iter"):
       # progreso al inicio (0%..)

        # ------------------------------------------------------------
        # (1) sample o^h (tu caso: o=phi)  -> obs_batch_by_pid[pid] shape (H,...)
        # ------------------------------------------------------------
        obs_batch_by_pid = sample_obs_batch_fn(
            df,
            H=H,
            seed=base_seed + t,
            plant_ids=pids,
            **sample_obs_batch_kwargs,
        )

        # ------------------------------------------------------------
        # (2) nets + Beta_t(o^h) con restricción beta > phi
        # ------------------------------------------------------------
        net_by_pid = build_nets_by_pid(theta_by_pid, input_dim=input_dim, hidden=hidden, device=device)

        Beta_t = np.zeros((H, len(pids)), dtype=float)
        for j, pid in enumerate(pids):
            Beta_t[:, j] = eval_policy_column_over_batch(
                net=net_by_pid[pid],
                theta=theta_by_pid[pid],
                obs_batch=obs_batch_by_pid[pid],
                device=device,
                min_bid=min_margin,
                floor_mode=floor_mode,
                reduce_mode=reduce_mode,
            )

        # ------------------------------------------------------------
        # (3) baseline eq/util con Beta_t
        # ------------------------------------------------------------
        eq_list_base = compute_eq_over_batch_from_Beta(
            template=template,
            Beta=Beta_t,
            pids=pids,
            compute_equilibrio_dispatch=compute_equilibrio_dispatch,
            pricing=pricing,
            unmet_price=unmet_price,
        )
        U_mean_base, U_by_h_base, out_by_plant_base = joint_utility_from_eq_list(
            eq_list_base,
            phi_batch_by_pid=obs_batch_by_pid,
            pricing=pricing,
        )

        # ------------------------------------------------------------
        # (4) ES grads por planta con oponentes fijos en Beta_t
        # ------------------------------------------------------------
        es_results: Dict[int, dict] = {}
        for pid in pids:
            u_mean_base_i = float(out_by_plant_base[pid]["u_mean"])
            es_results[pid] = perturb_single_agent_es_individual(
                planta=pid,
                theta_by_pid=theta_by_pid,
                net_by_pid=net_by_pid,
                obs_batch_by_pid=obs_batch_by_pid,
                phi_batch_by_pid=obs_batch_by_pid,
                template=template,
                Beta_tminus1=Beta_t,                      # oponentes fijos en t
                pids=pids,
                u_mean_base_planta=u_mean_base_i,
                compute_equilibrio_dispatch=compute_equilibrio_dispatch,
                joint_utility_from_eq_list=joint_utility_from_eq_list,
                pricing=pricing,
                unmet_price=unmet_price,
                P=P,
                sigma=sigma,
                seed=base_seed + 10_000 * t,              # desacopla seeds por iteración
                device=device,
                min_bid=min_margin,
                floor_mode=floor_mode,
                reduce_mode=reduce_mode,
            )

        # ------------------------------------------------------------
        # (5) update simultáneo: θ_{t+1} = θ_t + eta * grad_ES
        # ------------------------------------------------------------
        theta_next_by_pid: Dict[int, TensorDict] = {}
        for pid in pids:
            grad = es_results[pid]["grad_theta"]
            theta_prev = theta_by_pid[pid]
            theta_new: TensorDict = {}

            for k, v in theta_prev.items():
                if torch.is_tensor(v) and torch.is_floating_point(v) and (k in grad):
                    theta_new[k] = (v + float(eta) * grad[k].to(v.device)).detach().clone()
                elif torch.is_tensor(v):
                    theta_new[k] = v.detach().clone()
                else:
                    theta_new[k] = v

            theta_next_by_pid[pid] = theta_new

        theta_by_pid = theta_next_by_pid

        # ------------------------------------------------------------
        # (6) logging
        # ------------------------------------------------------------
        if store_history:
            beta_scalar_by_pid = {pid: float(np.mean(Beta_t[:, j])) for j, pid in enumerate(pids)}
            history["U_mean"].append(float(U_mean_base))
            history["beta_scalar_by_pid"].append(beta_scalar_by_pid)
            history["out_by_plant"].append(out_by_plant_base)
            history["es_results"].append(es_results)
            history["Beta"].append(Beta_t)

    out = {
        "theta_by_pid": theta_by_pid,   # θ_T
        "template": template,
        "pids": pids,
        "T": int(T),
        "H": int(H),
    }
    if store_history:
        out["history"] = history
    return out


c:\Users\HP\Documents\GitHub\TesisPEG\gpu_env_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
import numpy as np
import torch
import time
from typing import Dict, Any, List
from tqdm.auto import trange

TensorDict = Dict[str, torch.Tensor]

def run_NPGA(
    *,
    T: int,
    H: int,
    plant_ids: List[int],
    df,
    sample_obs_batch_fn,
    sample_obs_batch_kwargs: dict = None,
    build_template_fn,
    compute_equilibrio_dispatch,
    joint_utility_from_eq_list,
    pricing: str = "uniform",
    unmet_price: float = np.nan,
    eta: float = 1e-4,
    P: int = 64,
    sigma: float = 1e-3,
    seed: int = 123,
    device: str = "auto",
    input_dim: int = 1,
    hidden=(32, 64, 128, 128, 64, 32),
    min_margin: float = 1e-6,
    floor_mode: str = "max",
    reduce_mode: str = "max",
    theta_init_by_pid: Dict[int, TensorDict] = None,
    store_history: bool = True,
) -> dict:
    pids = [int(x) for x in plant_ids]
    sample_obs_batch_kwargs = sample_obs_batch_kwargs or {}

    template = build_template_fn(df)

    if theta_init_by_pid is None:
        theta_by_pid = init_theta0_by_pid_from_df(
            df,
            id_col="id_planta",
            price_col="precio_d",
            input_dim=input_dim,
            hidden=hidden,
            device=device,
        )
        theta_by_pid = {int(pid): theta_by_pid[int(pid)] for pid in pids}
    else:
        theta_by_pid = {int(pid): theta_init_by_pid[int(pid)] for pid in pids}

    history = {
      "U_mean": [],
      "beta_scalar_by_pid": [],
      "out_by_plant": [],
      "es_results": [],
      "Beta": [],
      "eq_list": [],        # <-- nuevo
      "phi_batch": []     # <-- nuevo
  } if store_history else None


    base_seed = int(seed)

    pbar = trange(int(T), desc="NPGA", unit="iter")
    t0 = time.time()

    for t in pbar:
        # (1) sample
        obs_batch_by_pid = sample_obs_batch_fn(
            df,
            H=H,
            seed=base_seed + t,
            plant_ids=pids,
            **sample_obs_batch_kwargs,
        )

        # (2) Beta_t
        net_by_pid = build_nets_by_pid(theta_by_pid, input_dim=input_dim, hidden=hidden, device=device)

        Beta_t = np.zeros((H, len(pids)), dtype=float)
        for j, pid in enumerate(pids):
            Beta_t[:, j] = eval_policy_column_over_batch(
                net=net_by_pid[pid],
                theta=theta_by_pid[pid],
                obs_batch=obs_batch_by_pid[pid],
                device=device,
                min_bid=min_margin,
                floor_mode=floor_mode,
                reduce_mode=reduce_mode,
            )

        # (3) baseline eq/util
        eq_list_base = compute_eq_over_batch_from_Beta(
            template=template,
            Beta=Beta_t,
            pids=pids,
            compute_equilibrio_dispatch=compute_equilibrio_dispatch,
            pricing=pricing,
            unmet_price=unmet_price,
        )
        U_mean_base, U_by_h_base, out_by_plant_base = joint_utility_from_eq_list(
            eq_list_base,
            phi_batch_by_pid=obs_batch_by_pid,
            pricing=pricing,
        )

        # (4) ES grads
        es_results: Dict[int, dict] = {}
        for pid in pids:
            u_mean_base_i = float(out_by_plant_base[pid]["u_mean"])
            es_results[pid] = perturb_single_agent_es_individual(
                planta=pid,
                theta_by_pid=theta_by_pid,
                net_by_pid=net_by_pid,
                obs_batch_by_pid=obs_batch_by_pid,
                phi_batch_by_pid=obs_batch_by_pid,
                template=template,
                Beta_tminus1=Beta_t,
                pids=pids,
                u_mean_base_planta=u_mean_base_i,
                compute_equilibrio_dispatch=compute_equilibrio_dispatch,
                joint_utility_from_eq_list=joint_utility_from_eq_list,
                pricing=pricing,
                unmet_price=unmet_price,
                P=P,
                sigma=sigma,
                seed=base_seed + 10_000 * t,
                device=device,
                min_bid=min_margin,
                floor_mode=floor_mode,
                reduce_mode=reduce_mode,
            )

        # (5) update
        theta_next_by_pid: Dict[int, TensorDict] = {}
        for pid in pids:
            grad = es_results[pid]["grad_theta"]
            theta_prev = theta_by_pid[pid]
            theta_new: TensorDict = {}
            for k, v in theta_prev.items():
                if torch.is_tensor(v) and torch.is_floating_point(v) and (k in grad):
                    theta_new[k] = (v + float(eta) * grad[k].to(v.device)).detach().clone()
                elif torch.is_tensor(v):
                    theta_new[k] = v.detach().clone()
                else:
                    theta_new[k] = v
            theta_next_by_pid[pid] = theta_new
        theta_by_pid = theta_next_by_pid

        # (6) logging
        if store_history:
            beta_scalar_by_pid = {pid: float(np.mean(Beta_t[:, j])) for j, pid in enumerate(pids)}
            history["U_mean"].append(float(U_mean_base))
            history["beta_scalar_by_pid"].append(beta_scalar_by_pid)
            history["out_by_plant"].append(out_by_plant_base)
            history["es_results"].append(es_results)
            history["Beta"].append(Beta_t)
            history["eq_list"].append(eq_list_base)           # aquí viven price24 y cantidades_despachadas
            history["phi_batch"].append(obs_batch_by_pid)

        # tiempo / ETA
        elapsed = time.time() - t0
        it_done = t + 1
        sec_per_it = elapsed / it_done
        eta_sec = sec_per_it * (int(T) - it_done)
        pbar.set_postfix_str(
            f"U={float(U_mean_base):.4g} | {sec_per_it:.2f}s/it | ETA={eta_sec:.0f}s"
        )

    out = {"theta_by_pid": theta_by_pid, "template": template, "pids": pids, "T": int(T), "H": int(H)}
    if store_history:
        out["history"] = history
    return out


In [23]:
import numpy as np
import pandas as pd

def extract_npga_panel_df(
    out: dict,
    *,
    pricing: str = "uniform",
    qty_key: str = "cantidades_despachadas",
    uniform_price_key: str = "price24",
    pab_price_keys=("price24", "precio24", "accepted_price24", "bid24"),
    phi_reduce: str = "max",   # si phi viene (H,d) y d!=24: "max" o "mean"
) -> pd.DataFrame:
    """
    Una fila por (t,h,id_planta) con listas de 24:
      - phi24
      - bid24           (si solo tienes bid escalar por escenario, se repite 24 veces)
      - q_dispatch24
      - price_paid24

    Requiere en out["history"]:
      - history["eq_list"]
      - history["phi_batch"]

    Para bid:
      - usa history["Beta"][t][h,j] si existe (bid escalar por escenario)
      - si no existe, intenta leer eq[pid]["bid24"] (si tu eq lo guarda)
      - si no existe nada, bid queda NaN
    """
    if "history" not in out or out["history"] is None:
        raise KeyError("out no tiene 'history' (usa store_history=True).")

    hist = out["history"]
    if "eq_list" not in hist:
        raise KeyError("history no tiene 'eq_list'. Guarda eq_list_base en history['eq_list'].")
    if "phi_batch" not in hist:
        raise KeyError("history no tiene 'phi_batch'. Guarda obs_batch_by_pid en history['phi_batch'].")

    pids = list(out["pids"])
    eq_hist = hist["eq_list"]
    phi_hist = hist["phi_batch"]
    beta_hist = hist.get("Beta", None)  # (H,I) por t, si lo guardaste

    pricing = pricing.lower().strip()
    rows = []

    T = len(eq_hist)
    for t in range(T):
        eq_list_t = eq_hist[t]
        phi_batch_t = phi_hist[t]

        H = len(eq_list_t)
        for h in range(H):
            eq = eq_list_t[h]

            # precio uniforme (clearing) por hora
            if pricing == "uniform":
                p24_uniform = np.asarray(eq[uniform_price_key], dtype=float).reshape(-1)
                if p24_uniform.size != 24:
                    raise ValueError(f"t={t}, h={h}: {uniform_price_key} size {p24_uniform.size}, esperaba 24")
            else:
                p24_uniform = None

            for j, pid in enumerate(pids):
                # cantidades despachadas
                q24 = np.asarray(eq[pid][qty_key], dtype=float).reshape(-1)
                if q24.size != 24:
                    raise ValueError(f"t={t}, h={h}, pid={pid}: {qty_key} size {q24.size}, esperaba 24")

                # phi24
                phi_arr = np.asarray(phi_batch_t[pid], dtype=float)
                if phi_arr.ndim == 1:
                    phi24 = np.repeat(phi_arr[h], 24)
                elif phi_arr.ndim == 2 and phi_arr.shape[1] == 24:
                    phi24 = phi_arr[h, :]
                elif phi_arr.ndim == 2:
                    v = phi_arr[h, :]
                    if phi_reduce == "mean":
                        phi24 = np.repeat(float(np.nanmean(v)), 24)
                    else:
                        phi24 = np.repeat(float(np.nanmax(v)), 24)
                else:
                    raise ValueError(f"t={t}, pid={pid}: phi shape no soportado {phi_arr.shape}")

                # bid (preferencia: history['Beta'] porque ahí está lo que usaste para simular)
                bid_scalar = np.nan
                bid24 = None

                if beta_hist is not None:
                    B = np.asarray(beta_hist[t], dtype=float)  # (H,I)
                    bid_scalar = float(B[h, j])
                    bid24 = np.repeat(bid_scalar, 24)
                else:
                    # si tu eq guarda un bid24 explícito por planta
                    if "bid24" in eq.get(pid, {}):
                        bid24 = np.asarray(eq[pid]["bid24"], dtype=float).reshape(-1)
                        if bid24.size != 24:
                            raise ValueError(f"t={t}, h={h}, pid={pid}: bid24 size {bid24.size}, esperaba 24")
                        bid_scalar = float(np.nanmean(bid24))
                    else:
                        bid24 = np.full(24, np.nan, dtype=float)

                # price_paid24
                if pricing == "uniform":
                    price24 = p24_uniform
                else:
                    price24 = None
                    for k in pab_price_keys:
                        if k in eq.get(pid, {}):
                            price24 = np.asarray(eq[pid][k], dtype=float).reshape(-1)
                            break

                    # fallback: si no hay precio por planta, en pay-as-bid lo más coherente es usar el bid
                    if price24 is None:
                        price24 = bid24

                    if price24.size != 24:
                        raise ValueError(f"t={t}, h={h}, pid={pid}: price vector size {price24.size}, esperaba 24")

                rows.append({
                    "t": int(t),
                    "h": int(h),
                    "id_planta": int(pid) if isinstance(pid, (np.integer,)) else pid,
                    "phi24": phi24.astype(float).tolist(),
                    "bid_scalar": float(bid_scalar),
                    #"bid24": bid24.astype(float).tolist(),
                    "q_dispatch24": q24.astype(float).tolist(),
                    "price_paid24": price24.astype(float).tolist(),
                })

    return pd.DataFrame(rows)



In [24]:

def _elemwise_mul_24(p_list, q_list):
    p = np.asarray(p_list, dtype=float).reshape(-1)
    q = np.asarray(q_list, dtype=float).reshape(-1)
    if p.size != 24 or q.size != 24:
        raise ValueError(f"Se esperaba len=24 en ambos. Got p={p.size}, q={q.size}")
    return (p * q).tolist()

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_compare_mean_payment_over_h_by_t(
    panel_uniform: pd.DataFrame,
    panel_pay_as_bid: pd.DataFrame,
    *,
    t_col: str = "t",
    h_col: str = "h",
    payment24_col: str = "payment24",
    payment_total_col: str = "payment_total",
    label_uniform: str = "uniform",
    label_pay_as_bid: str = "pay-as-bid",
    title: str = "Mean total payment per iteration (avg over scenarios h)",
):
    """
    Grafica dos líneas (uniform vs pay-as-bid):
      Para cada t: mean_h [ sum_plants sum_hours payment ]

    Requiere que cada panel tenga:
      - columnas t, h
      - y (payment24 lista len=24) o payment_total escalar.
    Retorna un DataFrame combinado con columnas:
      [t, mean_payment_over_h, mechanism]
    """

    def _agg(panel: pd.DataFrame, mechanism_label: str) -> pd.DataFrame:
        df = panel.copy()

        # payment_total por fila (t,h,id_planta)
        if payment_total_col not in df.columns:
            if payment24_col not in df.columns:
                raise KeyError(f"[{mechanism_label}] Falta '{payment24_col}' o '{payment_total_col}'.")
            df[payment_total_col] = df[payment24_col].apply(
                lambda x: float(np.nansum(np.asarray(x, dtype=float)))
            )

        # total por (t,h) sumando plantas
        pay_th = (
            df.groupby([t_col, h_col], as_index=False)[payment_total_col]
              .sum()
              .rename(columns={payment_total_col: "payment_th"})
        )

        # mean sobre h para cada t
        pay_t = (
            pay_th.groupby(t_col, as_index=False)["payment_th"]
                  .mean()
                  .rename(columns={"payment_th": "mean_payment_over_h"})
        )
        pay_t["mechanism"] = mechanism_label
        return pay_t

    u = _agg(panel_uniform, label_uniform)
    p = _agg(panel_pay_as_bid, label_pay_as_bid)

    combined = pd.concat([u, p], ignore_index=True)

    # plot
    plt.figure()
    for mech, g in combined.groupby("mechanism"):
        g = g.sort_values(t_col)
        plt.plot(g[t_col].to_numpy(), g["mean_payment_over_h"].to_numpy(), marker="o", label=mech)

    plt.xlabel(t_col)
    plt.ylabel("Mean total payment over h")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()

    return combined


# EJECUCION

PARAMETROS


In [26]:
# H=10
# P=10
# plant_ids = list(pd.unique(df["id_planta"].values))
# input_dim=24
# output_dim=1
# hidden=(32, 64, 128, 128, 64, 32)
# T=10
# seed=123
# eta=1e-4
# sigma=1e-3  #1/di
# pricing= "uniform"

PARTE 1

In [27]:
'''
phi_by_plant = sample_day_batches_phi(
  df,
  H=H,
  seed=seed,
  day_col="Fecha",
  id_col="id_planta",
  phi_col="phi",
  price_col="precio_d",
  plant_ids=plant_ids,
  replace=True,
  min_active_plants=1,
  pick="random",
)
'''

'\nphi_by_plant = sample_day_batches_phi(\n  df,\n  H=H,\n  seed=seed,\n  day_col="Fecha",\n  id_col="id_planta",\n  phi_col="phi",\n  price_col="precio_d",\n  plant_ids=plant_ids,\n  replace=True,\n  min_active_plants=1,\n  pick="random",\n)\n'

PARTE 2

PARTE 2.1

In [28]:
'''
# o_i^h ya sampleado: en tu caso o = phi
obs_batch_by_pid = phi_by_plant
# si phi es horario, típicamente input_dim=24 y obs_batch_by_pid[pid] tiene shape (H,24)

# t = 0  -> construye θ0 desde df y produce β0(o^h)
out0 = policies_and_beta_profile(t=0, df=df, obs_batch_by_pid=obs_batch_by_pid, input_dim=input_dim, device=device)
Beta0 = out0["Beta"]   # (H,I) = β0(o^h)

# t > 0 -> usa θ_{t-1} ya existente y produce β_{t-1}(o^h)
theta_prev_by_pid = out0["theta_by_pid"]  # ejemplo: aquí usarías las θ actualizadas del loop
out1 = policies_and_beta_profile(t=T, df=df, obs_batch_by_pid=obs_batch_by_pid,
                                 theta_prev_by_pid=theta_prev_by_pid,
                                 input_dim=input_dim, device=device)
Beta_tminus1 = out1["Beta"]  # (H,I) = β_{t-1}(o^h)
'''

'\n# o_i^h ya sampleado: en tu caso o = phi\nobs_batch_by_pid = phi_by_plant\n# si phi es horario, típicamente input_dim=24 y obs_batch_by_pid[pid] tiene shape (H,24)\n\n# t = 0  -> construye θ0 desde df y produce β0(o^h)\nout0 = policies_and_beta_profile(t=0, df=df, obs_batch_by_pid=obs_batch_by_pid, input_dim=input_dim, device=device)\nBeta0 = out0["Beta"]   # (H,I) = β0(o^h)\n\n# t > 0 -> usa θ_{t-1} ya existente y produce β_{t-1}(o^h)\ntheta_prev_by_pid = out0["theta_by_pid"]  # ejemplo: aquí usarías las θ actualizadas del loop\nout1 = policies_and_beta_profile(t=T, df=df, obs_batch_by_pid=obs_batch_by_pid,\n                                 theta_prev_by_pid=theta_prev_by_pid,\n                                 input_dim=input_dim, device=device)\nBeta_tminus1 = out1["Beta"]  # (H,I) = β_{t-1}(o^h)\n'

PARTE 2.2

In [29]:
'''
# ----------------------------
# 4) Uso (después de tener Beta_tminus1)
# ----------------------------
""""
Construye lo que no cambia con h (escenario).
Contiene:
template["I"]: lista/array de plantas (jugadores) en un orden fijo.

template["demanda"]: vector (24,) de demanda horaria.

template["_avail_mat"]: matriz (I,24) con la capacidad horaria available_q de cada planta, en el mismo orden que template["I"].
"""

template = build_template_from_df(df)

"""
pids es el orden de las columnas de Beta_tminus1.
Debe cumplir:la columna j de Beta_tminus1 corresponde a la planta pids[j].
"""
pids = plant_ids

"""
Toma Beta_tminus1 y construye un equilibrio por cada escenario h.
Para cada h:
arma juego_h copiando demanda y capacidades desde template,
mete los bids del escenario como juego_h[pid]["beta"] = Beta_tminus1[h, col(pid)],
llama compute_equilibrio_dispatch(juego_h, pricing=..., unmet_price=...),
guarda el resultado como eq_h.

Salida:
eq_list es una lista de longitud H.
eq_list[h] es el resultado del dispatch para el escenario h.
"""
eq_list = compute_eq_over_batch_from_Beta(
    template=template,
    Beta=Beta_tminus1,
    pids=pids,
    compute_equilibrio_dispatch=compute_equilibrio_dispatch,
    pricing="uniform",
    unmet_price=np.nan,
)
"""
Es el equilibrio del primer escenario del batch (h = 0).
Es un diccionario con:
eq_h0["price24"]: precios de mercado por hora (24,) para cada planta pid:
eq_h0[pid]["cantidades_despachadas"] (24,)
eq_h0[pid]["precio24"] (24,) pago por hora a esa planta
eq_h0[pid]["beta"] bid usado en ese escenario
"""
eq_h0 = eq_list[0] #Es el equilibrio del primer escenario del batch (h = 0)
'''

'\n# ----------------------------\n# 4) Uso (después de tener Beta_tminus1)\n# ----------------------------\n""""\nConstruye lo que no cambia con h (escenario).\nContiene:\ntemplate["I"]: lista/array de plantas (jugadores) en un orden fijo.\n\ntemplate["demanda"]: vector (24,) de demanda horaria.\n\ntemplate["_avail_mat"]: matriz (I,24) con la capacidad horaria available_q de cada planta, en el mismo orden que template["I"].\n"""\n\ntemplate = build_template_from_df(df)\n\n"""\npids es el orden de las columnas de Beta_tminus1.\nDebe cumplir:la columna j de Beta_tminus1 corresponde a la planta pids[j].\n"""\npids = plant_ids\n\n"""\nToma Beta_tminus1 y construye un equilibrio por cada escenario h.\nPara cada h:\narma juego_h copiando demanda y capacidades desde template,\nmete los bids del escenario como juego_h[pid]["beta"] = Beta_tminus1[h, col(pid)],\nllama compute_equilibrio_dispatch(juego_h, pricing=..., unmet_price=...),\nguarda el resultado como eq_h.\n\nSalida:\neq_list es una l

PARTE 3

In [30]:
'''
# ya tienes:
# - Beta_tminus1 (H,I)
# - template
# - pids
# - phi_batch_by_pid (tu obs_batch_by_pid porque o=phi y v=phi en tu simplificación)
# - compute_equilibrio_dispatch

eq_list = compute_eq_over_batch_from_Beta(
    template=template,
    Beta=Beta_tminus1,
    pids=pids,
    compute_equilibrio_dispatch=compute_equilibrio_dispatch,
    pricing="uniform",
    unmet_price=np.nan,
)

U_mean, U_by_h, out_by_plant = joint_utility_from_eq_list(
    eq_list,
    phi_batch_by_pid=obs_batch_by_pid,
    pricing="uniform",
)

# Esto es exactamente: ũ_{t-1} = (1/H) * sum_h ũ(v^h, β_{t-1}(o^h))
# donde ũ(v^h, β_{t-1}(o^h)) = sum_i u_i(h) = U_by_h[h]

#utilidad media por agente
'''

'\n# ya tienes:\n# - Beta_tminus1 (H,I)\n# - template\n# - pids\n# - phi_batch_by_pid (tu obs_batch_by_pid porque o=phi y v=phi en tu simplificación)\n# - compute_equilibrio_dispatch\n\neq_list = compute_eq_over_batch_from_Beta(\n    template=template,\n    Beta=Beta_tminus1,\n    pids=pids,\n    compute_equilibrio_dispatch=compute_equilibrio_dispatch,\n    pricing="uniform",\n    unmet_price=np.nan,\n)\n\nU_mean, U_by_h, out_by_plant = joint_utility_from_eq_list(\n    eq_list,\n    phi_batch_by_pid=obs_batch_by_pid,\n    pricing="uniform",\n)\n\n# Esto es exactamente: ũ_{t-1} = (1/H) * sum_h ũ(v^h, β_{t-1}(o^h))\n# donde ũ(v^h, β_{t-1}(o^h)) = sum_i u_i(h) = U_by_h[h]\n\n#utilidad media por agente\n'

PARTE 4

In [31]:
'''
# ============================================================
# 5) Nets por pid (solo arquitectura; θ se pasa como state_dict)
# ============================================================
planta_id=29
net_by_pid = {
    int(pid): PolicyNet(input_dim=input_dim, hidden=hidden, output_dim=output_dim).to(device)
    for pid in pids
}
planta_id = int(planta_id)
u_mean_base_planta = float(out_by_plant[planta_id]["u_mean"])

pid = int(planta_id)
beta_test = eval_policy_column_over_batch(
    net=net_by_pid[pid],
    theta=theta_prev_by_pid[pid],
    obs_batch=obs_batch_by_pid[pid],
    device=device,
    floor_mode="max",
)

print(beta_test.min(), beta_test.mean(), beta_test.max())

'''

'\n# ============================================================\n# 5) Nets por pid (solo arquitectura; θ se pasa como state_dict)\n# ============================================================\nplanta_id=29\nnet_by_pid = {\n    int(pid): PolicyNet(input_dim=input_dim, hidden=hidden, output_dim=output_dim).to(device)\n    for pid in pids\n}\nplanta_id = int(planta_id)\nu_mean_base_planta = float(out_by_plant[planta_id]["u_mean"])\n\npid = int(planta_id)\nbeta_test = eval_policy_column_over_batch(\n    net=net_by_pid[pid],\n    theta=theta_prev_by_pid[pid],\n    obs_batch=obs_batch_by_pid[pid],\n    device=device,\n    floor_mode="max",\n)\n\nprint(beta_test.min(), beta_test.mean(), beta_test.max())\n\n'

In [32]:
'''
# ============================================================
# 6) Perturbaciones ES por planta (una llamada por pid)
# ============================================================
res_i = perturb_single_agent_es_individual(
    planta=planta_id,
    theta_by_pid=theta_prev_by_pid,          # dict pid -> state_dict θ_{t-1}
    net_by_pid=net_by_pid,                   # dict pid -> PolicyNet
    obs_batch_by_pid=obs_batch_by_pid,       # dict pid -> o_i^h (phi) (H,) o (H,24)
    phi_batch_by_pid=obs_batch_by_pid,       # v_i^h (phi) (igual)
    template=template,
    Beta_tminus1=Beta_tminus1,               # (H,I) o (H,I,24)
    pids=pids,
    u_mean_base_planta=u_mean_base_planta,   # baseline individual
    compute_equilibrio_dispatch=compute_equilibrio_dispatch,
    joint_utility_from_eq_list=joint_utility_from_eq_list,
    pricing="uniform",
    unmet_price=np.nan,
    P=P,
    sigma=1e-3,
    seed=123,
    device=device,
)

# Ahora es_results[pid]["grad_theta"] es ∇_ES ũ respecto a θ_pid
  # esto alimenta tu update θ_i := θ_i + η * grad_theta_i
'''

'\n# ============================================================\n# 6) Perturbaciones ES por planta (una llamada por pid)\n# ============================================================\nres_i = perturb_single_agent_es_individual(\n    planta=planta_id,\n    theta_by_pid=theta_prev_by_pid,          # dict pid -> state_dict θ_{t-1}\n    net_by_pid=net_by_pid,                   # dict pid -> PolicyNet\n    obs_batch_by_pid=obs_batch_by_pid,       # dict pid -> o_i^h (phi) (H,) o (H,24)\n    phi_batch_by_pid=obs_batch_by_pid,       # v_i^h (phi) (igual)\n    template=template,\n    Beta_tminus1=Beta_tminus1,               # (H,I) o (H,I,24)\n    pids=pids,\n    u_mean_base_planta=u_mean_base_planta,   # baseline individual\n    compute_equilibrio_dispatch=compute_equilibrio_dispatch,\n    joint_utility_from_eq_list=joint_utility_from_eq_list,\n    pricing="uniform",\n    unmet_price=np.nan,\n    P=P,\n    sigma=1e-3,\n    seed=123,\n    device=device,\n)\n\n# Ahora es_results[pid]["grad_

hasta parte 4 ok toca parte 5

PARTE 5

In [33]:
'''
# =========================
# 2) Aplicar un paso NPGA para ESA planta
#    - actualiza θ_i
#    - recalcula β_i^t(o^h)
#    - inserta esa columna en Beta_t
# =========================
eta = 1e-4

theta_prev_by_pid, Beta_t, beta_t_planta = npga_step_single_agent(
    planta_id=planta_id,
    theta_by_pid=theta_prev_by_pid,   # se actualiza en sitio para esa planta
    net_by_pid=net_by_pid,
    obs_batch_by_pid=obs_batch_by_pid,
    Beta_tminus1=Beta_tminus1,
    pids=pids,
    res_i=res_i,                      # viene de perturb_single_agent_es_individual(...)
    eta=eta,
    device=device,
    min_margin=1e-6,
    floor_mode="max",
)

# Beta_t será el nuevo Beta_tminus1 en la siguiente iteración del loop
Beta_tminus1 = Beta_t
'''



'\n# =========================\n# 2) Aplicar un paso NPGA para ESA planta\n#    - actualiza θ_i\n#    - recalcula β_i^t(o^h)\n#    - inserta esa columna en Beta_t\n# =========================\neta = 1e-4\n\ntheta_prev_by_pid, Beta_t, beta_t_planta = npga_step_single_agent(\n    planta_id=planta_id,\n    theta_by_pid=theta_prev_by_pid,   # se actualiza en sitio para esa planta\n    net_by_pid=net_by_pid,\n    obs_batch_by_pid=obs_batch_by_pid,\n    Beta_tminus1=Beta_tminus1,\n    pids=pids,\n    res_i=res_i,                      # viene de perturb_single_agent_es_individual(...)\n    eta=eta,\n    device=device,\n    min_margin=1e-6,\n    floor_mode="max",\n)\n\n# Beta_t será el nuevo Beta_tminus1 en la siguiente iteración del loop\nBeta_tminus1 = Beta_t\n'

PARTE 6

In [34]:
# ============================================================
# 3) Modo de uso dentro del loop t
# ============================================================
# t-1: tienes theta_by_pid, net_by_pid, obs_batch_by_pid, template
# 1) ES simultáneo (oponentes fijos en t-1)
'''
out_es = run_es_for_all_plants(
    plant_ids=plant_ids,
    theta_by_pid=theta_prev_by_pid,          # θ_{t-1}
    net_by_pid=net_by_pid,
    obs_batch_by_pid=obs_batch_by_pid,
    template=template,
    compute_equilibrio_dispatch=compute_equilibrio_dispatch,
    joint_utility_from_eq_list=joint_utility_from_eq_list,
    pricing="uniform",
    unmet_price=np.nan,
    P=P,
    sigma=sigma,
    seed=seed,
    device=device,
    min_margin=1e-6,
    floor_mode="max",
)

# 2) aplicar updates simultáneos y construir Beta_t (será el nuevo Beta_tminus1)
out_upd = apply_es_updates_for_all_plants(
    plant_ids=plant_ids,
    theta_by_pid=theta_prev_by_pid,          # θ_{t-1}
    es_results=out_es["es_results"],         # grads por planta
    eta=eta,
    net_by_pid=net_by_pid,
    obs_batch_by_pid=obs_batch_by_pid,
    template=template,
    compute_equilibrio_dispatch=compute_equilibrio_dispatch,
    joint_utility_from_eq_list=joint_utility_from_eq_list,
    pricing="uniform",
    unmet_price=np.nan,
    device=device,
    min_margin=1e-6,
    floor_mode="max",
)

# 3) avanzar a t:
theta_prev_by_pid = out_upd["theta_by_pid"]   # θ_t
Beta_tminus1      = out_upd["Beta_t"]         # β_t(o^h)  (input para el siguiente periodo)
pids              = out_upd["pids"]
'''


'\nout_es = run_es_for_all_plants(\n    plant_ids=plant_ids,\n    theta_by_pid=theta_prev_by_pid,          # θ_{t-1}\n    net_by_pid=net_by_pid,\n    obs_batch_by_pid=obs_batch_by_pid,\n    template=template,\n    compute_equilibrio_dispatch=compute_equilibrio_dispatch,\n    joint_utility_from_eq_list=joint_utility_from_eq_list,\n    pricing="uniform",\n    unmet_price=np.nan,\n    P=P,\n    sigma=sigma,\n    seed=seed,\n    device=device,\n    min_margin=1e-6,\n    floor_mode="max",\n)\n\n# 2) aplicar updates simultáneos y construir Beta_t (será el nuevo Beta_tminus1)\nout_upd = apply_es_updates_for_all_plants(\n    plant_ids=plant_ids,\n    theta_by_pid=theta_prev_by_pid,          # θ_{t-1}\n    es_results=out_es["es_results"],         # grads por planta\n    eta=eta,\n    net_by_pid=net_by_pid,\n    obs_batch_by_pid=obs_batch_by_pid,\n    template=template,\n    compute_equilibrio_dispatch=compute_equilibrio_dispatch,\n    joint_utility_from_eq_list=joint_utility_from_eq_list,\n    

PARTE 7

In [35]:
# # t=0
# out = run_NPGA_all(
#     t=0,
#     plant_ids=plant_ids,
#     df=df,
#     obs_batch_by_pid=obs_batch_by_pid,
#     template=template,
#     compute_equilibrio_dispatch=compute_equilibrio_dispatch,
#     joint_utility_from_eq_list=joint_utility_from_eq_list,
#     pricing="uniform",
#     eta=1e-4,
#     P=64,
#     sigma=1e-3,
#     device=device,
#     input_dim=input_dim,
#     hidden=hidden,
# )


In [36]:
H=117
P=64
plant_ids = list(pd.unique(df["id_planta"].values))
input_dim=24
output_dim=1
hidden = (10, 10)
T=100
seed=123
eta=1e-4
sigma=1e-3  #1/di
pricing= "uniform"

In [37]:
out = run_NPGA(
    T=T,
    H=H,
    plant_ids=plant_ids,
    df=df,
    sample_obs_batch_fn=sample_day_batches_phi,
    sample_obs_batch_kwargs={
        "day_col": "Fecha",
        "id_col": "id_planta",
        "phi_col": "phi",
        "price_col": "precio_d",
        "replace": True,
        "min_active_plants": 1,
        "pick": "random",
    },
    build_template_fn=build_template_from_df,
    compute_equilibrio_dispatch=compute_equilibrio_dispatch,
    joint_utility_from_eq_list=joint_utility_from_eq_list,
    pricing=pricing,
    eta=eta,
    P=P,
    sigma=sigma,
    seed=seed,
    device=device,
    input_dim=input_dim,
    hidden=hidden,
    min_margin=1e-6,
    floor_mode="max",
    reduce_mode="max",
)

theta_final = out["theta_by_pid"]
U_path = out["history"]["U_mean"]
beta_path_t0 = out["history"]["beta_scalar_by_pid"][0]   # dict pid -> beta promedio en H


NPGA: 100%|██████████| 100/100 [9:40:00<00:00, 348.01s/iter, U=8139 | 348.01s/it | ETA=0s]          


In [52]:
import torch
import json

# Guardar el modelo aparte
torch.save(out["theta_by_pid"], "theta_by_pid.pth")

# Reemplazar en el JSON por el nombre del archivo
out_copy = out.copy()
out_copy["theta_by_pid"] = "theta_by_pid.pth"

with open("out.json", "w") as f:
    json.dump(out_copy, f, indent=4)


TypeError: Object of type ndarray is not JSON serializable

In [38]:
device

device(type='cuda')

In [39]:
out.keys()

dict_keys(['theta_by_pid', 'template', 'pids', 'T', 'H', 'history'])

In [40]:
out['history'].keys()

dict_keys(['U_mean', 'beta_scalar_by_pid', 'out_by_plant', 'es_results', 'Beta', 'eq_list', 'phi_batch'])

In [41]:
#out['history']['beta_scalar_by_pid']

In [42]:

def _elemwise_mul_24(p_list, q_list):
    p = np.asarray(p_list, dtype=float).reshape(-1)
    q = np.asarray(q_list, dtype=float).reshape(-1)
    if p.size != 24 or q.size != 24:
        raise ValueError(f"Se esperaba len=24 en ambos. Got p={p.size}, q={q.size}")
    return (p * q).tolist()

In [43]:
#uniform
if pricing =="uniform":
  df_uniform= pd.DataFrame([out['history']['beta_scalar_by_pid'][-1]])
  panel_uniform = extract_npga_panel_df(out, pricing="uniform")   # o "pay_as_bid"
  panel_uniform["payment24"] = panel_uniform.apply(
    lambda r: _elemwise_mul_24(r["price_paid24"], r["q_dispatch24"]),
    axis=1
).apply(lambda x: float(np.nansum(x)))
  import os

  out_dir = "t100"
  os.makedirs(out_dir, exist_ok=True)

  panel_uniform.to_csv(f"{out_dir}/panel_uniformAsymetric.csv", index=False)


In [44]:
#pay as bid
if pricing =="pay_as_bid":
  df_discriminatory = pd.DataFrame([out['history']['beta_scalar_by_pid'][-1]])
  panel_pay_as_bid = extract_npga_panel_df(out, pricing="pay_as_bid")   # o "pay_as_bid"
  panel_pay_as_bid["payment24"] = panel_pay_as_bid.apply(
    lambda r: _elemwise_mul_24(r["price_paid24"], r["q_dispatch24"]),
    axis=1
).apply(lambda x: float(np.nansum(x)))
  import os

  out_dir = "t100"
  os.makedirs(out_dir, exist_ok=True)


  panel_pay_as_bid.to_csv(f"{out_dir}/panel_pay_as_bidAsymetric.csv", index=False)



In [46]:
df_mean = df.groupby('id_planta')['precio_d'].mean().reset_index()
df_mean.index = df_mean.index + 1
df_mean.T



,1,2,3,4,5,6,7,8,9,10,...,27,28,29,30,31,32,33,34,35,36
id_planta,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,8.000000,9.0000,10.000000,...,27.000000,28.000000,29.000000,30.000000,31.000000,32.000000,33.000000,34.000000,35.000000,36.000000
precio_d,0.844274,1.597018,1.594619,0.685118,0.299241,0.962952,0.321918,0.297995,0.4098,0.302745,...,0.944758,1.189166,0.650867,0.231679,0.133024,0.787619,0.408832,0.502125,0.391574,0.340464


In [1]:
panel_uniform

NameError: name 'panel_uniform' is not defined

In [54]:
import torch
import numpy as np
from collections.abc import Mapping, Sequence

def sanitize_for_torchsave(x):
    if torch.is_tensor(x):
        return x.detach().cpu()
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, torch.nn.Module):
        return x.state_dict()
    if isinstance(x, Mapping):
        return {k: sanitize_for_torchsave(v) for k, v in x.items()}
    if isinstance(x, Sequence) and not isinstance(x, (str, bytes, bytearray)):
        return type(x)(sanitize_for_torchsave(v) for v in x)
    return x  # números, strings, None, etc.

ckpt = sanitize_for_torchsave(out)
torch.save(ckpt, "out_ckpt.pt")



history = sanitize_for_torchsave(out.get("history", {}))
config  = out.get("config", {})

torch.save({"history": history, "config": config}, "meta.pt")


In [50]:
df_uniform

,1,2,3,4,5,6,7,8,9,10,...,27,28,29,30,31,32,33,34,35,36
0,1.699485,3.395908,3.433571,1.488953,0.295143,2.025183,1.115517,1.037295,0.572856,0.243663,...,2.004878,2.321641,1.865994,0.443153,0.232303,1.918515,1.156217,0.774561,0.754725,0.531122


In [56]:
import os, json
import pandas as pd

os.makedirs(out_dir, exist_ok=True)

obj = out["history"]["beta_scalar_by_pid"][-1]

# 1) CSV (si ya es DataFrame, se guarda directo; si es dict/list, se convierte)
if isinstance(obj, pd.DataFrame):
    df = obj
elif isinstance(obj, list):
    df = pd.DataFrame(obj)              # lista de dicts / filas
elif isinstance(obj, dict):
    df = pd.DataFrame([obj])            # dict -> una fila (cols = keys)
else:
    raise TypeError(f"Tipo no soportado: {type(obj)}")

df.to_csv(os.path.join(out_dir, "dfpay_as_bidAsymetric.csv"), index=False)

# 2) JSON (raw del objeto + JSON tabular del DataFrame)
with open(os.path.join(out_dir, "dfpay_as_bidAsymetric_raw.json"), "w", encoding="utf-8") as f:
    json.dump(obj, f, ensure_ascii=False)

df.to_json(os.path.join(out_dir, "dfpay_as_bidAsymetric.json"), orient="records")



In [ ]:
out['history']['beta_scalar_by_pid'][-1].to_csv(f"{out_dir}/ddbidAsymetric.csv", index=False)

## plot del costo del subastador = expenditure's cost

In [47]:
cmp_df = plot_compare_mean_payment_over_h_by_t(panel_uniform, panel_pay_as_bid)


NameError: name 'panel_pay_as_bid' is not defined

In [ ]:
cmp_df.to_csv(f"{out_dir}/comparacionMeanPaymentAsymmetricT50.csv", index=False)

plot del markup de las plantas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_compare_mean_bid_by_t(
    panel_uniform: pd.DataFrame,
    panel_pay_as_bid: pd.DataFrame,
    *,
    t_col: str = "t",
    bid24_col: str = "bid24",
    bid_scalar_col: str = "bid_scalar",
    label_uniform: str = "uniform",
    label_pay_as_bid: str = "pay-as-bid",
    title: str = "Mean bid per iteration",
):
    """
    Grafica dos líneas (uniform vs pay-as-bid):
      mean bid por iteración t, promediando sobre (h, planta, hora).

    Acepta bid como:
      - bid24: lista len=24, o
      - bid_scalar: escalar por (t,h,planta).

    Retorna DF combinado: [t, mean_bid, mechanism].
    """

    def _mean_bid_from_panel(panel: pd.DataFrame, mech: str) -> pd.DataFrame:
        df = panel.copy()

        if bid24_col in df.columns:
            # promedio sobre horas dentro de cada fila, luego promedio global por t
            df["_bid_row_mean"] = df[bid24_col].apply(lambda x: float(np.nanmean(np.asarray(x, dtype=float))))
        elif bid_scalar_col in df.columns:
            df["_bid_row_mean"] = pd.to_numeric(df[bid_scalar_col], errors="coerce")
        else:
            raise KeyError(f"[{mech}] Falta '{bid24_col}' o '{bid_scalar_col}' en el panel.")

        out = (
            df.groupby(t_col, as_index=False)["_bid_row_mean"]
              .mean()
              .rename(columns={"_bid_row_mean": "mean_bid"})
        )
        out["mechanism"] = mech
        return out

    u = _mean_bid_from_panel(panel_uniform, label_uniform)
    p = _mean_bid_from_panel(panel_pay_as_bid, label_pay_as_bid)
    combined = pd.concat([u, p], ignore_index=True)

    plt.figure()
    for mech, g in combined.groupby("mechanism"):
        g = g.sort_values(t_col)
        plt.plot(g[t_col].to_numpy(), g["mean_bid"].to_numpy(), marker="o", label=mech)

    plt.xlabel(t_col)
    plt.ylabel("Mean bid")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()

    return combined


In [ ]:
bid_cmp = plot_compare_mean_bid_by_t(panel_uniform, panel_pay_as_bid)


In [ ]:
bid_cmp.to_csv(f"{out_dir}/comparacionMeanBidAsymmetricT50.csv", index=False)

## markup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_compare_mean_markup_winners_by_t(
    panel_uniform: pd.DataFrame,
    panel_pay_as_bid: pd.DataFrame,
    *,
    t_col: str = "t",
    h_col: str = "h",
    phi24_col: str = "phi24",
    bid24_col: str = "bid24",
    bid_scalar_col: str = "bid_scalar",
    payment24_col: str = "payment24",
    label_uniform: str = "uniform",
    label_pay_as_bid: str = "pay-as-bid",
    title: str = "Mean markup of winners per iteration",
):
    """
    Markup = bid - phi, SOLO para ganadores.
    Ganador por hora si payment24[hour] > 0.

    Si bid24 no existe, lo construye como repeat(bid_scalar, 24).

    Robusto a que phi24/payment24 vengan como:
      - escalar
      - vector len=24
      - vector len=H (se toma elemento h y se repite 24)
      - matriz (H,24) (se toma fila h)

    Para cada (t,h): promedio del markup sobre (plantas, horas) ganadoras.
    Luego para cada t: mean sobre h.
    Grafica dos líneas: uniform vs pay-as-bid.

    Retorna DF combinado: [t, mean_markup_over_h, mechanism].
    """

    def _as_list24(x, h: int) -> list:
        a = np.asarray(x, dtype=float)

        if a.ndim == 0:  # escalar
            return [float(a)] * 24

        if a.ndim == 1:
            if a.size == 24:
                return a.tolist()
            # si viene como (H,) (por escenario): toma h y repite 24
            if a.size > 24:
                return [float(a[int(h)])] * 24
            raise ValueError(f"Vector len={a.size} no compatible con 24.")

        if a.ndim == 2:
            if a.shape[1] == 24:
                return a[int(h), :].tolist()
            raise ValueError(f"Matriz shape={a.shape} no compatible con (_,24).")

        raise ValueError(f"ndim={a.ndim} no soportado.")

    def _ensure_bid24(df: pd.DataFrame, mech: str) -> pd.DataFrame:
        df = df.copy()
        if bid24_col not in df.columns:
            if bid_scalar_col not in df.columns:
                raise KeyError(f"[{mech}] Falta '{bid24_col}' y '{bid_scalar_col}'.")
            df[bid_scalar_col] = pd.to_numeric(df[bid_scalar_col], errors="coerce")
            df[bid24_col] = df[bid_scalar_col].apply(
                lambda b: [float(b)] * 24 if np.isfinite(b) else [np.nan] * 24
            )
        return df

    def _sanitize_24_cols(df: pd.DataFrame, mech: str) -> pd.DataFrame:
        df = df.copy()
        if h_col not in df.columns:
            raise KeyError(f"[{mech}] Falta columna '{h_col}'.")
        # fuerza listas len=24 usando h por fila
        for c in (phi24_col, payment24_col, bid24_col):
            if c in df.columns:
                df[c] = df.apply(lambda r: _as_list24(r[c], r[h_col]), axis=1)
        return df

    def _agg(panel: pd.DataFrame, mech: str) -> pd.DataFrame:
        df = panel.copy()
        if phi24_col not in df.columns:
            raise KeyError(f"[{mech}] Falta columna '{phi24_col}'.")
        if payment24_col not in df.columns:
            raise KeyError(f"[{mech}] Falta columna '{payment24_col}'.")

        df = _ensure_bid24(df, mech)
        df = _sanitize_24_cols(df, mech)

        def row_winner_markup_mean(r):
            phi = np.asarray(r[phi24_col], dtype=float)
            bid = np.asarray(r[bid24_col], dtype=float)
            pay = np.asarray(r[payment24_col], dtype=float)

            winners = (pay > 0) & np.isfinite(phi) & np.isfinite(bid)
            if not np.any(winners):
                return np.nan
            return float(np.nanmean(bid[winners] - phi[winners]))

        df["_markup_w_mean"] = df.apply(row_winner_markup_mean, axis=1)

        day = (
            df.groupby([t_col, h_col], as_index=False)["_markup_w_mean"]
              .mean()
              .rename(columns={"_markup_w_mean": "markup_day_mean"})
        )

        out = (
            day.groupby(t_col, as_index=False)["markup_day_mean"]
               .mean()
               .rename(columns={"markup_day_mean": "mean_markup_over_h"})
        )
        out["mechanism"] = mech
        return out

    u = _agg(panel_uniform, label_uniform)
    p = _agg(panel_pay_as_bid, label_pay_as_bid)
    combined = pd.concat([u, p], ignore_index=True)

    plt.figure()
    for mech, g in combined.groupby("mechanism"):
        g = g.sort_values(t_col)
        plt.plot(g[t_col].to_numpy(), g["mean_markup_over_h"].to_numpy(), marker="o", label=mech)

    plt.xlabel(t_col)
    plt.ylabel("Mean markup (bid - phi), winners only")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()

    return combined


In [ ]:


markup_cmp = plot_compare_mean_markup_winners_by_t(panel_uniform, panel_pay_as_bid)

# markup_cmp queda con columnas: ['t', 'mean_markup_over_h', 'mechanism']
markup_cmp.tail()


In [ ]:
markup_cmp.to_csv(f"{out_dir}/comparacionMeanMarkupAsymmetricT50.csv", index=False)

## nota que al final soloo importa es el ultimo t que es cuando ya se entrenó